In [ ]:
%%capture
!pip install distfit
!pip install statsmodels
!pip install miceforest



In [ ]:
%%capture
!pip install sdv==1.29.1 rdt==1.18.2

In [ ]:
import sdv, rdt
print("SDV:", sdv.__version__)
print("RDT:", rdt.__version__)

SDV: 1.29.1
RDT: 1.18.2


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
from scipy.stats import gaussian_kde
import scipy.stats as st
from distfit import distfit
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
import numpy as np
import plotly.graph_objects as go
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from statsmodels.nonparametric.smoothers_lowess import lowess
import miceforest as mf
from scipy.stats import lognorm
from scipy.stats import ks_2samp
from scipy.stats import chisquare
from scipy.stats import chi2_contingency
from scipy.spatial.distance import jensenshannon
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
import statsmodels.api as sm
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from scipy.stats import dweibull, genextreme, norm, lognorm, t
from scipy.spatial.distance import jensenshannon
from scipy.stats import kstest
from sdmetrics.reports.single_table import QualityReport
from sdv.metadata import SingleTableMetadata
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score


###Paso 1: Análisis descriptivo y exploratorio de los datos

Este paso es esencial, ya que para replicar el comportamiento de los datos es necesario conocerlo primero.

####Lectura de datos: ICFES: mejores saber pro de 2016 a 2020

In [ ]:
urls=[['1dDCj447NxMty-Y7TY4YV7-N1d3-xDaZ9',"1871784120"],['1hC3Tv8mnevEGhN6QXrl417N9iKUKSaJM',"971686572"],['1DqlNGo_PSLDTzJEyJQ69WBYz-w1wmLRS',"1960156222"],
      ['1T1H5Doeg3egvx5HXYxOSghGsyhsoDzeI',"1311313853"],['1tqv9abnQx-pqzWmQSSGMRiUKPlpf1Zi9',"1962302517"]]
       ##lista con ids para acceder a los data frames sin tener que cargar drive

In [ ]:
dfs=[]#lista vacia, en este se guardarán los data frames.

In [ ]:
for x in range(0, len(urls)): #cargamos los dataframes
    url = f"https://docs.google.com/spreadsheets/d/{urls[x][0]}/export?format=csv&gid={urls[x][1]}"
    dfs.append(pd.read_csv(url))

In [ ]:
filas=[]
for x in range(0, len(urls)):
  filas.append(dfs[x].shape[0])
columnas=[]
for x in range(0, len(urls)):
  columnas.append(dfs[x].shape[1])
Total_datos=[]
for x in range(0, len(urls)):
  Total_datos.append(filas[x]*columnas[x])
datos_faltantes=[]
for x in range(0, len(urls)):
  datos_faltantes.append(dfs[x].isna().sum().sum())
porc_faltantes=[]
for x in range(0, len(urls)):
  porc_faltantes.append(round((datos_faltantes[x]/Total_datos[x])*100,2))
necesita_imputacion=[]
for x in range(0, len(urls)):
  if datos_faltantes[x]>0:
    necesita_imputacion.append('Sí')
  else:
    necesita_imputacion.append('No')

#### Dimensiones, datos totales y datos faltantes.

In [ ]:
#covariable años
for x in range(0, len(dfs)-4):#Estandarizamos el nombre de las columnas al data frame del año 2016.
      dfs[x].columns=dfs[1].columns
años=list(range(2016,2016+len(dfs)))
años
diagnostico=pd.DataFrame({'Año de presentación':años,'Total de filas':filas,'Total de columnas':columnas,'Total de datos':Total_datos,'Datos faltantes':datos_faltantes,
                         '% de datos faltantes':porc_faltantes,'¿Necesita imputación de datos?':necesita_imputacion})
diagnostico

,Año de presentación,Total de filas,Total de columnas,Total de datos,Datos faltantes,% de datos faltantes,¿Necesita imputación de datos?
0,2016,11033,16,176528,2756,1.56,Sí
1,2017,10763,16,172208,108,0.06,Sí
2,2018,10166,16,162656,796,0.49,Sí
3,2019,10691,16,171056,802,0.47,Sí
4,2020,12712,16,203392,1676,0.82,Sí


In [ ]:
data_icfes = {
    "Año": [2016, 2017, 2018, 2019, 2020],
    "Evaluados": [245162, 248855, 240403, 269431, 253114],
    "Inscritos": [248669, 251740, 243569, 272667, 258306],
    "% participación": [98.6, 98.9, 98.7, 98.8, 98.0]
}

data_icfes = pd.DataFrame(data_icfes)
print(data_icfes)

    Año  Evaluados  Inscritos  % participación
0  2016     245162     248669             98.6
1  2017     248855     251740             98.9
2  2018     240403     243569             98.7
3  2019     269431     272667             98.8
4  2020     253114     258306             98.0


In [ ]:
propor_mejores = {
    "Año": [2016, 2017, 2018, 2019, 2020],
    "Evaluados": [245162, 248855, 240403, 269431, 253114],
    "Mejores": diagnostico.loc[0:4,'Total de filas']
}
propor_mejores = pd.DataFrame(propor_mejores)
propor_mejores["% mejores"] = (propor_mejores["Mejores"] / propor_mejores["Evaluados"]) * 100
print(propor_mejores)
print(propor_mejores["% mejores"].mean())
propor_mejores.to_latex(
    index=True,
    caption="Total estudiantes evaluados y seleccionados entre los mejor",
    label="% mejores",
    longtable=False,
    escape=False)

    Año  Evaluados  Mejores  % mejores
0  2016     245162    11033   4.500290
1  2017     248855    10763   4.325009
2  2018     240403    10166   4.228733
3  2019     269431    10691   3.967992
4  2020     253114    12712   5.022243
4.40885309538663


'\\begin{table}\n\\caption{Total estudiantes evaluados y seleccionados entre los mejor}\n\\label{% mejores}\n\\begin{tabular}{lrrrr}\n\\toprule\n & Año & Evaluados & Mejores & % mejores \\\\\n\\midrule\n0 & 2016 & 245162 & 11033 & 4.500290 \\\\\n1 & 2017 & 248855 & 10763 & 4.325009 \\\\\n2 & 2018 & 240403 & 10166 & 4.228733 \\\\\n3 & 2019 & 269431 & 10691 & 3.967992 \\\\\n4 & 2020 & 253114 & 12712 & 5.022243 \\\\\n\\bottomrule\n\\end{tabular}\n\\end{table}\n'

In [ ]:
tipos_faltantes_16_20=pd.DataFrame({'Tipo':dfs[0].dtypes,'Faltantes 2016':dfs[0].isna().sum(),'Faltantes 2017':dfs[1].isna().sum(),
                              'Faltantes 2018':dfs[2].isna().sum(),'Faltantes 2019':dfs[3].isna().sum(),'Faltantes 2020':dfs[4].isna().sum()})
tipos_faltantes_16_20

,Tipo,Faltantes 2016,Faltantes 2017,Faltantes 2018,Faltantes 2019,Faltantes 2020
APELLIDOS Y NOMBRES,object,0.0,0.0,0.0,0.0,0.0
DEPARTAMENTO_INSTITUCION,object,0.0,0.0,0.0,0.0,0.0
GRUPO REFERENCIA,object,0.0,0.0,0.0,0.0,0.0
MUNICIPIO_INSTITUCION,object,0.0,0.0,0.0,0.0,0.0
NOMBRE DE LA INSTITUCION,object,0.0,0.0,0.0,0.0,0.0
NOMBRE DEL PROGRAMA ACADEMICO,object,0.0,0.0,0.0,0.0,0.0
PERCENTIL COMPETENCIAS CIUDAD,NaN,NaN,NaN,81.0,81.0,216.0
PERCENTIL COMPETENCIAS CIUDADANAS,float64,291.0,14.0,NaN,NaN,NaN
PERCENTIL COMUNICACIÓN ESCRIT,NaN,NaN,NaN,0.0,0.0,0.0
PERCENTIL COMUNICACIÓN ESCRITA,int64,0.0,0.0,NaN,NaN,NaN


--------------------------------------------------------------------------------

# V categóricas

In [ ]:
#Unir datasets 2016-2020
año=2016
for i in dfs[0:5]:
  i["año"]=año
  año=año+1
Hist_2016_2020 = pd.concat(dfs[0:5], ignore_index=False)
Hist_2016_2020.head()

,APELLIDOS Y NOMBRES,GRUPO REFERENCIA,NOMBRE DEL PROGRAMA ACADEMICO,DEPARTAMENTO_INSTITUCION,MUNICIPIO_INSTITUCION,NOMBRE DE LA INSTITUCION,COMUNICACIÓN ESCRITA,COMUNICACIÓN ESCRITA_2,RAZONAMIENTO CUANTITATIVO,RAZONAMIENTO CUANTITATIVO_2,...,PUNTAJE LECTURA CRÍTICA,PERCENTIL COMPETENCIAS CIUDADANAS,PUNTAJE COMPETENCIAS CIUDADANAS,PERCENTIL INGLÉS,PUNTAJE INGLÉS,PERCENTIL COMPETENCIAS CIUDAD,PUNTAJE COMPETENCIAS CIUDADAN,PERCENTIL COMUNICACIÓN ESCRIT,PERCENTIL RAZONAMIENTO CUANTI,PUNTAJE RAZONAMIENTO CUANTITA
0,MOYANO CHAPARRO LAURA MARIA,CIENCIAS SOCIALES,CIENCIA POLITICA,BOGOTA,BOGOTÁ D.C.,UNIVERSIDAD NACIONAL DE COLOMBIA-BOGOTÁ D.C.,98.0,212.0,100.0,220.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,MORALES BASTO JUAN PABLO,SALUD,NUTRICION Y DIETETICA,BOGOTA,BOGOTÁ D.C.,UNIVERSIDAD NACIONAL DE COLOMBIA-BOGOTÁ D.C.,99.0,215.0,95.0,199.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,LOPEZ QUESADA SANTIAGO,CIENCIAS AGROPECUARIAS,INGENIERIA AGRONOMICA,BOGOTA,BOGOTÁ D.C.,UNIVERSIDAD NACIONAL DE COLOMBIA-BOGOTÁ D.C.,97.0,207.0,84.0,180.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SUAREZ NIÑO ANDRES JAVIER,ECONOMIA,ECONOMIA,SANTANDER,BUCARAMANGA,UNIVERSIDAD INDUSTRIAL DE SANTANDER-BUCARAMANGA,93.0,195.0,100.0,228.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,MARTINEZ MARTINEZ CARLOS ANDRES,CIENCIAS SOCIALES,CIENCIA POLITICA Y GOBIERNO,ATLANTICO,BARRANQUILLA,UNIVERSIDAD DEL NORTE-BARRANQUILLA,97.0,207.0,83.0,180.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
##Simulación variables categóricas
categóricas=Hist_2016_2020.iloc[:,1:6].reset_index(drop=True)#selección de filas
categóricas.head()

,GRUPO REFERENCIA,NOMBRE DEL PROGRAMA ACADEMICO,DEPARTAMENTO_INSTITUCION,MUNICIPIO_INSTITUCION,NOMBRE DE LA INSTITUCION
0,CIENCIAS SOCIALES,CIENCIA POLITICA,BOGOTA,BOGOTÁ D.C.,UNIVERSIDAD NACIONAL DE COLOMBIA-BOGOTÁ D.C.
1,SALUD,NUTRICION Y DIETETICA,BOGOTA,BOGOTÁ D.C.,UNIVERSIDAD NACIONAL DE COLOMBIA-BOGOTÁ D.C.
2,CIENCIAS AGROPECUARIAS,INGENIERIA AGRONOMICA,BOGOTA,BOGOTÁ D.C.,UNIVERSIDAD NACIONAL DE COLOMBIA-BOGOTÁ D.C.
3,ECONOMIA,ECONOMIA,SANTANDER,BUCARAMANGA,UNIVERSIDAD INDUSTRIAL DE SANTANDER-BUCARAMANGA
4,CIENCIAS SOCIALES,CIENCIA POLITICA Y GOBIERNO,ATLANTICO,BARRANQUILLA,UNIVERSIDAD DEL NORTE-BARRANQUILLA


In [ ]:
categóricas_df=pd.DataFrame({"Variable":categóricas.columns,"% Datos reales":0.0*len(categóricas.columns),"Chi2":0.0*len(categóricas.columns),"Jensen-Shannon":0.0*len(categóricas.columns),
                             "Hellinger":0.0*len(categóricas.columns)})
for i,categoría in enumerate(categóricas.columns):
  original = categóricas[categoría].value_counts(normalize=True)
  ct_simulada = np.random.choice(
      original.index,
      size=len(categóricas[categoría]),
      replace=True,
      p=original.values)
  #sim_categóricas[categoría]=ct_simulada
  simulada_counts = pd.Series(ct_simulada).value_counts()
  original_counts = categóricas[categoría].value_counts()
  simulada_counts, original_counts = simulada_counts.align(original_counts, fill_value=0)
  #Filtracions
  categóricas_df.loc[i, "% Datos reales"]=((categóricas[categoría]==ct_simulada).sum()/len(ct_simulada))*100
  #Chi2
  tabla = np.array([original_counts, simulada_counts])
  chi2, p, dof, expected = chi2_contingency(tabla)
  categóricas_df.loc[i, "Chi2"]=p
  #Jensen-Shannon
  p = original_counts / original_counts.sum()
  q = simulada_counts / simulada_counts.sum()
  js = jensenshannon(p, q)
  categóricas_df.loc[i, "Jensen-Shannon"]=js
  #Hellinger
  hellinger = np.sqrt(0.5 * ((np.sqrt(p) - np.sqrt(q))**2).sum())
  categóricas_df.loc[i, "Hellinger"]=hellinger
categóricas_df

,Variable,% Datos reales,Chi2,Jensen-Shannon,Hellinger
0,GRUPO REFERENCIA,12.572925,0.982892,0.006839,0.006842
1,NOMBRE DEL PROGRAMA ACADEMICO,2.920618,1.000000,0.041260,0.043337
2,DEPARTAMENTO_INSTITUCION,17.713357,0.852708,0.009357,0.009649
3,MUNICIPIO_INSTITUCION,21.177639,1.000000,0.012514,0.013378
4,NOMBRE DE LA INSTITUCION,2.991059,1.000000,0.026226,0.027054


In [ ]:
# @title Corrección: simulación condicional
df = categóricas.copy()
n = len(df)

# Función para construir mapa condicional P(target | given)
def get_conditional_map(df, given, target):
    mp = {}
    for key, g in df.groupby(given):
        vc = g[target].value_counts(normalize=True)
        mp[key] = (vc.index.to_numpy(), vc.to_numpy())
    return mp

# Construir jerarquía de distribuciones condicionales
P_dept = df['DEPARTAMENTO_INSTITUCION'].value_counts(normalize=True)

P_mun_dept  = get_conditional_map(df, 'DEPARTAMENTO_INSTITUCION', 'MUNICIPIO_INSTITUCION')
P_inst_mun  = get_conditional_map(df, 'MUNICIPIO_INSTITUCION', 'NOMBRE DE LA INSTITUCION')
P_prog_inst = get_conditional_map(df, 'NOMBRE DE LA INSTITUCION', 'NOMBRE DEL PROGRAMA ACADEMICO')
P_grup_prog = get_conditional_map(df, 'NOMBRE DEL PROGRAMA ACADEMICO', 'GRUPO REFERENCIA')


def safe_choice(key, lookup):
    """Escoge un valor respetando distribución condicional, evitando errores"""
    if key not in lookup:
        return "OTRO"
    cats, probs = lookup[key]

    # Convertir a numpy array por seguridad
    cats = np.array(cats)
    probs = np.array(probs, dtype=float)

    # Evitar NaN
    probs = np.nan_to_num(probs, nan=0.0)

    # Si no suman 1, normalizar
    s = probs.sum()
    if s == 0:
        return "OTRO"
    probs = probs / s

    # Evitar errores de shape
    if len(cats) != len(probs):
        return "OTRO"

    return np.random.choice(cats, p=probs)


# 1. Departamento
dept_sim = np.random.choice(P_dept.index, size=n, p=P_dept.values)

# 2. Municipio | Departamento
mun_sim = [ safe_choice(d, P_mun_dept)  for d in dept_sim ]

# 3. Institución | Municipio
inst_sim = [ safe_choice(m, P_inst_mun)  for m in mun_sim ]

# 4. Programa | Institución
prog_sim = [ safe_choice(i, P_prog_inst) for i in inst_sim ]

# 5. Grupo | Programa
grup_sim = [ safe_choice(p, P_grup_prog) for p in prog_sim ]

# Construir dataframe sintético
sim_df = pd.DataFrame({
    'DEPARTAMENTO_INSTITUCION': dept_sim,
    'MUNICIPIO_INSTITUCION': mun_sim,
    'NOMBRE DE LA INSTITUCION': inst_sim,
    'NOMBRE DEL PROGRAMA ACADEMICO': prog_sim,
    'GRUPO REFERENCIA': grup_sim
})

sim_df.head()

,DEPARTAMENTO_INSTITUCION,MUNICIPIO_INSTITUCION,NOMBRE DE LA INSTITUCION,NOMBRE DEL PROGRAMA ACADEMICO,GRUPO REFERENCIA
0,ATLANTICO,BARRANQUILLA,UNIVERSIDAD DEL NORTE-BARRANQUILLA,INGENIERIA MECANICA,INGENIERÍA
1,BOGOTÁ,BOGOTÁ D.C.,UNIVERSIDAD DE LOS ANDES-BOGOTÁ D.C.,ADMINISTRACION DE EMPRESAS,ADMINISTRACIÓN Y AFINES
2,ANTIOQUIA,MEDELLÍN,UNIVERSIDAD DE ANTIOQUIA-MEDELLIN,DERECHO,DERECHO
3,VALLE,CALI,UNIVERSIDAD AUTONOMA DE OCCIDENTE-CALI,CONTADURIA PUBLICA,CONTADURÍA Y AFINES
4,BOGOTÁ,BOGOTÁ D.C.,"UNIVERSIDAD DISTRITAL""FRANCISCO JOSE DE CALDAS...",INGENIERIA FORESTAL,INGENIERÍA


In [ ]:
categóricas=Hist_2016_2020.iloc[:,1:6].reset_index(drop=True)#selección de filas
categóricas_df=pd.DataFrame({"Variable":categóricas.columns,"% Datos reales":0.0*len(categóricas.columns),"Chi2":0.0*len(categóricas.columns),"Jensen-Shannon":0.0*len(categóricas.columns),
                             "Hellinger":0.0*len(categóricas.columns)})
for i,categoría in enumerate(categóricas.columns):
  simulada_counts = sim_df[categoría].value_counts()
  original_counts = categóricas[categoría].value_counts()
  simulada_counts, original_counts = simulada_counts.align(original_counts, fill_value=0)
  #Filtracions
  categóricas_df.loc[i, "% Datos reales"]=(categóricas[categoría]==sim_df[categoría]).sum()/len(sim_df[categoría])*100
  #Chi2
  tabla = np.array([original_counts, simulada_counts])
  chi2, p, dof, expected = chi2_contingency(tabla)
  categóricas_df.loc[i, "Chi2"]=p
  #Jensen-Shannon
  p = original_counts / original_counts.sum()
  q = simulada_counts / simulada_counts.sum()
  js = jensenshannon(p, q)
  categóricas_df.loc[i, "Jensen-Shannon"]=js
  #Hellinger
  hellinger = np.sqrt(0.5 * ((np.sqrt(p) - np.sqrt(q))**2).sum())
  categóricas_df.loc[i, "Hellinger"]=hellinger
categóricas_df

,Variable,% Datos reales,Chi2,Jensen-Shannon,Hellinger
0,GRUPO REFERENCIA,12.336314,0.880956,0.008123,0.008132
1,NOMBRE DEL PROGRAMA ACADEMICO,3.007315,1.000000,0.044698,0.047081
2,DEPARTAMENTO_INSTITUCION,17.700713,0.998728,0.006545,0.006546
3,MUNICIPIO_INSTITUCION,21.208345,0.999978,0.013596,0.014337
4,NOMBRE DE LA INSTITUCION,3.103043,1.000000,0.027972,0.029064


In [ ]:
#Propensión
# ---- 1. Construir dataframe combinado ----
categóricas['Real']=[1]*len(categóricas['GRUPO REFERENCIA'])
sim_df['Real']=[0]*len(sim_df['GRUPO REFERENCIA'])
dk=pd.concat([categóricas,sim_df],ignore_index=True)
dk

# ---- 2. One-hot encode la variable categórica ----
enc = OneHotEncoder(sparse_output=False)
X = enc.fit_transform(dk[['GRUPO REFERENCIA', 'NOMBRE DEL PROGRAMA ACADEMICO',
       'MUNICIPIO_INSTITUCION','DEPARTAMENTO_INSTITUCION',
       'NOMBRE DE LA INSTITUCION']])
y = dk['Real']

# ---- 3. Entrenar clasificador ----
clf = RandomForestClassifier(n_estimators=200, random_state=0)
clf.fit(X, y)

# ---- 4. Obtener propensión (P(real)) ----
dk['propension'] = clf.predict_proba(X)[:, 1]

# ---- 5. Ver resumen ----
print(dk.groupby('Real')['propension'].describe())

        count      mean       std       min       25%       50%       75%  \
Real                                                                        
0     55365.0  0.478076  0.103606  0.000000  0.449349  0.493579  0.526177   
1     55365.0  0.521653  0.095659  0.086884  0.472663  0.509474  0.552930   

          max  
Real           
0     0.92221  
1     1.00000  


23 mins

In [ ]:
categóricas.drop(columns=['Real'],inplace=True)

In [ ]:
#TVAES
from sdv.single_table import TVAESynthesizer
from sdv.metadata import Metadata

metadata = Metadata.detect_from_dataframe(categóricas)

model = TVAESynthesizer(
    metadata,
    epochs=300,
    batch_size=100,
    enforce_rounding=True,
    enforce_min_max_values=True
)

model.fit(categóricas)

synthetic_TVAE = model.sample(len(categóricas))
synthetic_TVAE.head()
"-------------------------------------------"
#Propensión
# ---- 1. Construir dataframe combinado ----
categóricas['Real']=[1]*len(categóricas['GRUPO REFERENCIA'])
synthetic_TVAE['Real']=[0]*len(synthetic_TVAE['GRUPO REFERENCIA'])
dk=pd.concat([categóricas,synthetic_TVAE],ignore_index=True)
dk

# ---- 2. One-hot encode la variable categórica ----
enc = OneHotEncoder(sparse_output=False)
X = enc.fit_transform(dk[['GRUPO REFERENCIA', 'NOMBRE DEL PROGRAMA ACADEMICO',
       'MUNICIPIO_INSTITUCION','DEPARTAMENTO_INSTITUCION',
       'NOMBRE DE LA INSTITUCION']])
y = dk['Real']

# ---- 3. Entrenar clasificador ----
clf = RandomForestClassifier(n_estimators=200, random_state=0)
clf.fit(X, y)

# ---- 4. Obtener propensión (P(real)) ----
dk['propension'] = clf.predict_proba(X)[:, 1]

# ---- 5. Ver resumen ----
print(dk.groupby('Real')['propension'].describe())


/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/ctgan/synthesizers/_utils.py:16: FutureWarning: `cuda` parameter is deprecated and will be removed in a future release. Please use `enable_gpu` instead.
  warnings.warn(


        count      mean       std       min       25%       50%       75%  \
Real                                                                        
0     55365.0  0.283445  0.282681  0.000000  0.007510  0.185419  0.544047   
1     55365.0  0.727728  0.191658  0.059808  0.610764  0.746081  0.874006   

           max  
Real            
0     0.988771  
1     1.000000  


2horas

In [ ]:
#cópulas gaussianas
from sdv.single_table import GaussianCopulaSynthesizer
from sdv.metadata import SingleTableMetadata

# Detectar metadata desde el dataframe real
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(data=categóricas)
metadata.validate()

# Crear modelo Gaussian Copula
model = GaussianCopulaSynthesizer(
    metadata=metadata,
    enforce_min_max_values=False
)

# Ajustar el modelo con el MISMO dataframe usado en metadata
model.fit(categóricas)

# Generar el mismo número de filas que el dataframe original
synthetic = model.sample(num_rows=len(categóricas))

synthetic.head()
"-------------------------------------------"
#Propensión
# ---- 1. Construir dataframe combinado ----
categóricas['Real']=[1]*len(categóricas['GRUPO REFERENCIA'])
synthetic['Real']=[0]*len(synthetic['GRUPO REFERENCIA'])
dk=pd.concat([categóricas,synthetic],ignore_index=True)
dk

# ---- 2. One-hot encode la variable categórica ----
enc = OneHotEncoder(sparse_output=False)
X = enc.fit_transform(dk[['GRUPO REFERENCIA', 'NOMBRE DEL PROGRAMA ACADEMICO',
       'MUNICIPIO_INSTITUCION','DEPARTAMENTO_INSTITUCION',
       'NOMBRE DE LA INSTITUCION']])
y = dk['Real']

# ---- 3. Entrenar clasificador ----
clf = RandomForestClassifier(n_estimators=200, random_state=0)
clf.fit(X, y)

# ---- 4. Obtener propensión (P(real)) ----
dk['propension'] = clf.predict_proba(X)[:, 1]

# ---- 5. Ver resumen ----
print(dk.groupby('Real')['propension'].describe())





/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:168: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


        count     mean       std       min  25%  50%       75%       max
Real                                                                    
0     55365.0  0.02274  0.090620  0.000000  0.0  0.0  0.014608  0.994937
1     55365.0  0.98938  0.029989  0.334446  1.0  1.0  1.000000  1.000000


In [ ]:
#pasar a excel
synthetic.to_excel('synthetic_categóricas.xlsx', index=False)
synthetic_TVAE.to_excel('synthetic_TVAE.xlsx', index=False)
sim_df.to_excel('simuladas_arbol.xlsx', index=False)

In [ ]:
#fast-MD
import numpy as np
import pandas as pd

np.random.seed(42)
# Función FAST-MD extendida para categóricas
def generate_synthetic_mixed(df, n_samples):
    synthetic = pd.DataFrame()
    for col in categóricas.columns:
        if df[col].dtype == 'object' or categóricas[col].dtype.name == 'category':
            # Muestreo de categorías según distribución de frecuencias
            freqs = df[col].value_counts(normalize=True)
            synthetic[col] = np.random.choice(freqs.index, size=n_samples, p=freqs.values)
        else:
            # Numéricas: normal aproximada
            mu, sigma = df[col].mean(), df[col].std()
            synthetic[col] = np.random.normal(mu, sigma, n_samples)
    return synthetic

synthetic_dp= generate_synthetic_mixed(categóricas, n_samples=1000)
def max_discrepancy(real, synth):
    max_disc = 0
    for col in real.columns:
        stat, _ = ks_2samp(real[col], synth[col])
        max_disc = max(max_disc, stat)
    return max_disc

md_score = max_discrepancy(categóricas, synthetic_dp)
print(f"Maximum Discrepancy entre real y sintético: {md_score:.4f}")
"----------------------------------------"
#Propensión
# ---- 1. Construir dataframe combinado ----
categóricas['Real']=[1]*len(categóricas['GRUPO REFERENCIA'])
synthetic_dp['Real']=[0]*len(synthetic_dp['GRUPO REFERENCIA'])
dk=pd.concat([categóricas,synthetic_dp],ignore_index=True)
dk

# ---- 2. One-hot encode la variable categórica ----
enc = OneHotEncoder(sparse_output=False)
X = enc.fit_transform(dk[['GRUPO REFERENCIA', 'NOMBRE DEL PROGRAMA ACADEMICO',
       'MUNICIPIO_INSTITUCION','DEPARTAMENTO_INSTITUCION',
       'NOMBRE DE LA INSTITUCION']])
y = dk['Real']

# ---- 3. Entrenar clasificador ----
clf = RandomForestClassifier(n_estimators=200, random_state=0)
clf.fit(X, y)

# ---- 4. Obtener propensión (P(real)) ----
dk['propension'] = clf.predict_proba(X)[:, 1]

# ---- 5. Ver resumen ----
print(dk.groupby('Real')['propension'].describe())


Maximum Discrepancy entre real y sintético: 0.0331
        count      mean       std   min    25%   50%       75%       max
Real                                                                    
0      1000.0  0.152564  0.096810  0.01  0.095  0.14  0.194677  0.993459
1     55365.0  0.999646  0.003181  0.83  1.000  1.00  1.000000  1.000000


# Variables cuantitativas

In [ ]:
Hist_2016_2020.drop(columns=['APELLIDOS Y NOMBRES'], inplace=True)

In [ ]:
# --------------------------------------------------------
# 1. Importar librerías
# --------------------------------------------------------
from sdv.single_table import GaussianCopulaSynthesizer
from sdv.metadata import SingleTableMetadata
from sdmetrics.reports.single_table import QualityReport


# --------------------------------------------------------
# 2. Variables que deseas evaluar
# --------------------------------------------------------
vars_puntajes = [
    'PUNTAJE LECTURA CRÍTICA',
    'PUNTAJE COMUNICACIÓN ESCRITA',
    'PUNTAJE RAZONAMIENTO CUANTITATIVO',
    'PUNTAJE COMPETENCIAS CIUDADANAS',
    'PUNTAJE INGLÉS'
]


# --------------------------------------------------------
# 3. Crear metadata y validarla
# --------------------------------------------------------
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(data=Hist_2016_2020)
metadata.validate()


# --------------------------------------------------------
# 4. Crear y entrenar el modelo Gaussian Copula
# --------------------------------------------------------
gaussian_copula_model = GaussianCopulaSynthesizer(
    metadata=metadata,
    enforce_min_max_values=False
)

gaussian_copula_model.fit(Hist_2016_2020)


# --------------------------------------------------------
# 5. Generar datos sintéticos
# --------------------------------------------------------
N_muestras = len(Hist_2016_2020)
synthetic = gaussian_copula_model.sample(num_rows=N_muestras)


# --------------------------------------------------------
# 6. Construir metadata filtrada para evaluar solo las columnas seleccionadas
# --------------------------------------------------------
metadata_dict = metadata.to_dict()

filtered_columns = {
    col: metadata_dict['columns'][col]
    for col in vars_puntajes
}

filtered_metadata_dict = metadata_dict.copy()
filtered_metadata_dict['columns'] = filtered_columns


# --------------------------------------------------------
# 7. Generar Quality Report
# --------------------------------------------------------
report = QualityReport()

report.generate(
    real_data=Hist_2016_2020[vars_puntajes],
    synthetic_data=synthetic[vars_puntajes],
    metadata=filtered_metadata_dict
)


# --------------------------------------------------------
# 8. Mostrar métricas: Column Shapes (fidelidad marginal)
# --------------------------------------------------------
details_shapes = report.get_details(property_name='Column Shapes')
print("\n===== FIDELIDAD MARGINAL (Column Shapes) =====\n")
print(details_shapes.sort_values(by='Score', ascending=True))


# --------------------------------------------------------
# 9. Mostrar métricas: Column Pair Trends (fidelidad de correlación)
# --------------------------------------------------------
details_corr = report.get_details(property_name='Column Pair Trends')
print("\n===== FIDELIDAD DE CORRELACIÓN (Column Pair Trends) =====\n")
print(details_corr.sort_values(by='Score', ascending=True))



/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:168: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 5/5 [00:00<00:00, 39.55it/s]|
Column Shapes Score: 87.53%

(2/2) Evaluating Column Pair Trends: |██████████| 10/10 [00:00<00:00, 49.18it/s]|
Column Pair Trends Score: 97.01%

Overall Score (Average): 92.27%


===== FIDELIDAD MARGINAL (Column Shapes) =====

                              Column        Metric     Score
2  PUNTAJE RAZONAMIENTO CUANTITATIVO  KSComplement  0.790428
3    PUNTAJE COMPETENCIAS CIUDADANAS  KSComplement  0.791394
1       PUNTAJE COMUNICACIÓN ESCRITA  KSComplement  0.912735
4                     PUNTAJE INGLÉS  KSComplement  0.921874
0            PUNTAJE LECTURA CRÍTICA  KSComplement  0.959932

===== FIDELIDAD DE CORRELACIÓN (Column Pair Trends) =====

                            Column 1                           Column 2  \
2            PUNTAJE LECTURA CRÍTICA    PUNTAJE COMPETENCIAS CIUDADANAS   
1            PUNTAJE LECTURA CRÍTICA  PUNTAJE RAZONAMIENTO CUANTITATIVO   
9    PUNTAJE COMPETENCIA

In [ ]:
from sdmetrics.single_table import LogisticDetection

## --- Validación de Propensión (Evasión) ---
score_detection = LogisticDetection.compute(
    real_data=Hist_2016_2020,
    synthetic_data=synthetic
)
propension_evasion = 1 - score_detection

print("\n--- Resultados de Propensión a Evasión ---")
print(f"Propensión a Evasión (Fidelidad para Engañar): {propension_evasion:.2%}")
print(f"Detection Score (Capacidad de Distinción): {score_detection:.2%}")

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c


--- Resultados de Propensión a Evasión ---
Propensión a Evasión (Fidelidad para Engañar): 31.24%
Detection Score (Capacidad de Distinción): 68.76%


In [ ]:
## --- Variables de Puntaje ---
vars_puntajes = [
    'PUNTAJE LECTURA CRÍTICA',
    'PUNTAJE COMUNICACIÓN ESCRITA',
    'PUNTAJE RAZONAMIENTO CUANTITATIVO',
    'PUNTAJE COMPETENCIAS CIUDADANAS',
    'PUNTAJE INGLÉS'
]

# --- 2. Inicialización del DataFrame de Resultados ---
import pandas as pd
import numpy as np
from scipy.stats import kstest
from scipy.spatial.distance import jensenshannon

# ... (Definiciones de df_final y sim_df, y numéricas_df) ...

# SOLUCIÓN: INICIALIZACIÓN DEL DATAFRAME
numéricas_df = pd.DataFrame({
    "Variable": vars_puntajes,
    "KSComplemnt": 0.0,
    "Jensen_Shannon": 0.0,
    "Hellinger": 0.0
})

N_BINS = 10
bins = np.linspace(0, 500, N_BINS + 1)


# --- 3. Bucle de Comparación (CORREGIDO) ---
for i, columna in enumerate(vars_puntajes):

    # CORRECCIÓN CRÍTICA: Asegurar la misma longitud y limpiar NaN
    # Usamos .dropna() para excluir cualquier NaN y asegurarnos de que solo comparamos
    # la población completa de datos. Si los DataFrames tienen tamaños diferentes
    # DESPUÉS de la imputación, es necesario alinearlos. Asumo que se debe a NaNs.
    data_original = Hist_2016_2020[columna].dropna()
    data_simulada = synthetic[columna].dropna()

    # Si aún fallan, es porque el número de filas en df_final y sim_df es estructuralmente diferente.
    # Si quieres comparar solo N elementos, podrías usar:
    min_len = min(len(data_original), len(data_simulada))
    data_original = data_original.iloc[:min_len]
    data_simulada = data_simulada.iloc[:min_len]

    # A. Test de Kolmogorov-Smirnov (KS)
    # Se aplica directamente a los datos continuos y limpios.
    ks_stat, ks_p_value = kstest(data_simulada, data_original)
    numéricas_df.loc[i, "KSComplemnt"] = 1 - ks_stat

    # B. Discretización (Binning) para JSD y Hellinger
    # Convertimos a distribuciones de frecuencia, usando los datos limpios.
    original_bins = pd.cut(data_original, bins=bins, include_lowest=True, duplicates='drop')
    imputada_bins = pd.cut(data_simulada, bins=bins, include_lowest=True, duplicates='drop')

    original_counts = original_bins.value_counts()
    imputada_counts = imputada_bins.value_counts()

    # Alineación
    imputada_counts, original_counts = imputada_counts.align(original_counts, fill_value=0)

    # Normalización para obtener las probabilidades (p y q)
    p = original_counts / original_counts.sum()
    q = imputada_counts / imputada_counts.sum()

    # C. Distancia de Jensen-Shannon (JSD)
    js = jensenshannon(p, q)
    numéricas_df.loc[i, "Jensen_Shannon"] = js

    # D. Distancia de Hellinger (HD)
    hellinger = np.sqrt(0.5 * ((np.sqrt(p) - np.sqrt(q))**2).sum())
    numéricas_df.loc[i, "Hellinger"] = hellinger

numérica5 = numéricas_df
numérica5

,Variable,KSComplemnt,Jensen_Shannon,Hellinger
0,PUNTAJE LECTURA CRÍTICA,0.959918,0.027230,0.027636
1,PUNTAJE COMUNICACIÓN ESCRITA,0.912704,0.096576,0.099350
2,PUNTAJE RAZONAMIENTO CUANTITATIVO,0.789903,0.122107,0.131824
3,PUNTAJE COMPETENCIAS CIUDADANAS,0.791628,0.183900,0.188154
4,PUNTAJE INGLÉS,0.921792,0.025335,0.028270


In [ ]:
numérica5.to_latex()

'\\begin{tabular}{llrrr}\n\\toprule\n & Variable & KSComplemnt & Jensen_Shannon & Hellinger \\\\\n\\midrule\n0 & PUNTAJE LECTURA CRÍTICA & 0.959918 & 0.027230 & 0.027636 \\\\\n1 & PUNTAJE COMUNICACIÓN ESCRITA & 0.912704 & 0.096576 & 0.099350 \\\\\n2 & PUNTAJE RAZONAMIENTO CUANTITATIVO & 0.789903 & 0.122107 & 0.131824 \\\\\n3 & PUNTAJE COMPETENCIAS CIUDADANAS & 0.791628 & 0.183900 & 0.188154 \\\\\n4 & PUNTAJE INGLÉS & 0.921792 & 0.025335 & 0.028270 \\\\\n\\bottomrule\n\\end{tabular}\n'

In [ ]:
#@title CopulaGAN
from sdv.single_table import CopulaGANSynthesizer # ¡Cambiado a CopulaGAN!
from sdv.metadata import SingleTableMetadata

# --- Configuración de Metadata ---
metadata = SingleTableMetadata()
col_categoricas=['GRUPO REFERENCIA',
        'NOMBRE DEL PROGRAMA ACADEMICO', 'DEPARTAMENTO_INSTITUCION',
        'MUNICIPIO_INSTITUCION', 'NOMBRE DE LA INSTITUCION']

# Iteración sobre todas las columnas del DataFrame Hist_2016_2020
for col in Hist_2016_2020.columns:
    if col in col_categoricas:
        metadata.add_column(col, sdtype='categorical')
    else:
        # Suponemos que las demás son numéricas (incluyendo los puntajes)
        metadata.add_column(col, sdtype='numerical')

metadata.validate()
print("Metadata validada. Tipos de datos listos.")


# --- Inicialización del Modelo CopulaGAN ---
# NOTA: Este entrenamiento será SIGNIFICATIVAMENTE más lento que GaussianCopula.
model = CopulaGANSynthesizer(
    metadata=metadata,
    enforce_min_max_values=False # Se mantiene la opción para truncamiento manual
    # Puedes añadir un argumento como epochs=1000 si deseas forzar más entrenamiento.
)

print("Iniciando entrenamiento con CopulaGANSynthesizer...")
model.fit(Hist_2016_2020)

# --- Generación de Datos Sintéticos ---
synthetic_gan = model.sample(55365)

print("Datos sintéticos generados con CopulaGAN.")
synthetic_gan.head()

In [ ]:
## 1. Obtener la metadata completa (como hiciste antes)
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(data=Hist_2016_2020)
metadata_dict = metadata.to_dict()

# 2. CREAR UN NUEVO DICCIONARIO DE METADATA FILTRADO
# El nuevo diccionario mantiene la clave 'columns' y solo incluye las columnas necesarias.

filtered_columns = {col: metadata_dict['columns'][col] for col in vars_puntajes}

filtered_metadata_dict = metadata_dict.copy()
filtered_metadata_dict['columns'] = filtered_columns # Reemplazamos las columnas con el subconjunto

# 3. INSTANCIAR EL OBJETO REPORT (SOLUCIÓN AL ERROR NameError)
report = QualityReport()

# 3. Generar reporte con la metadata filtrada
report.generate(
    real_data=Hist_2016_2020[vars_puntajes], # IMPORTANTE: Filtrar también los datos
    synthetic_data=synthetic_gan[vars_puntajes],
    metadata=filtered_metadata_dict
)
# Muestra los detalles de cada métrica
details = report.get_details(property_name='Column Shapes')
print("Detalles de Column Shapes (Fidelidad Marginal):")
print(details.sort_values(by='Score', ascending=True))

details_corr = report.get_details(property_name='Column Pair Trends')
print("\nDetalles de Column Pair Trends (Fidelidad de Correlación):")
print(details_corr.sort_values(by='Score', ascending=True))

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 5/5 [00:00<00:00, 36.50it/s]|
Column Shapes Score: 90.56%

(2/2) Evaluating Column Pair Trends: |██████████| 10/10 [00:00<00:00, 66.88it/s]|
Column Pair Trends Score: 96.81%

Overall Score (Average): 93.68%

Detalles de Column Shapes (Fidelidad Marginal):
                              Column        Metric     Score
0            PUNTAJE LECTURA CRÍTICA  KSComplement  0.842354
4                     PUNTAJE INGLÉS  KSComplement  0.884338
3    PUNTAJE COMPETENCIAS CIUDADANAS  KSComplement  0.900786
2  PUNTAJE RAZONAMIENTO CUANTITATIVO  KSComplement  0.944392
1       PUNTAJE COMUNICACIÓN ESCRITA  KSComplement  0.955981

Detalles de Column Pair Trends (Fidelidad de Correlación):
                            Column 1                           Column 2  \
5       PUNTAJE COMUNICACIÓN ESCRITA    PUNTAJE COMPETENCIAS CIUDADANAS   
6       PUNTAJE COMUNICACIÓN ESCRITA                     PUNTAJE INGLÉS   
3            PUNTAJE LECT

In [ ]:
## --- Validación de Propensión (Evasión) ---
score_detection = LogisticDetection.compute(
    real_data=Hist_2016_2020,
    synthetic_data=synthetic_gan
)
propension_evasion = 1 - score_detection

print("\n--- Resultados de Propensión a Evasión ---")
print(f"Propensión a Evasión (Fidelidad para Engañar): {propension_evasion:.2%}")
print(f"Detection Score (Capacidad de Distinción): {score_detection:.2%}")

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c


--- Resultados de Propensión a Evasión ---
Propensión a Evasión (Fidelidad para Engañar): 60.77%
Detection Score (Capacidad de Distinción): 39.23%


In [ ]:
##
data=pd.read_csv("/content/df_copula_simulado_truncado.csv")
data.head()

,PUNTAJE LECTURA CRÍTICA,PUNTAJE COMUNICACIÓN ESCRITA,PUNTAJE RAZONAMIENTO CUANTITATIVO,PUNTAJE COMPETENCIAS CIUDADANAS,PUNTAJE INGLÉS,PUNTAJE GLOBAL
0,298.0,232.0,184.0,228.0,200.0,213.654247
1,298.0,197.0,191.0,217.0,211.0,225.027424
2,188.0,186.0,186.0,184.0,172.0,188.308857
3,188.0,165.0,195.0,172.0,199.0,184.830600
4,163.0,191.0,161.0,162.0,203.0,175.545499


In [ ]:
## --- Variables de Puntaje ---
vars_puntajes = [
    'PUNTAJE LECTURA CRÍTICA',
    'PUNTAJE COMUNICACIÓN ESCRITA',
    'PUNTAJE RAZONAMIENTO CUANTITATIVO',
    'PUNTAJE COMPETENCIAS CIUDADANAS',
    'PUNTAJE INGLÉS'
]

# --- 2. Inicialización del DataFrame de Resultados ---
import pandas as pd
import numpy as np
from scipy.stats import kstest
from scipy.spatial.distance import jensenshannon

# ... (Definiciones de df_final y sim_df, y numéricas_df) ...

# SOLUCIÓN: INICIALIZACIÓN DEL DATAFRAME
numéricas_df = pd.DataFrame({
    "Variable": vars_puntajes,
    "KSComplemnt": 0.0,
    "Jensen_Shannon": 0.0,
    "Hellinger": 0.0
})

N_BINS = 10
bins = np.linspace(0, 500, N_BINS + 1)


# --- 3. Bucle de Comparación (CORREGIDO) ---
for i, columna in enumerate(vars_puntajes):

    # CORRECCIÓN CRÍTICA: Asegurar la misma longitud y limpiar NaN
    # Usamos .dropna() para excluir cualquier NaN y asegurarnos de que solo comparamos
    # la población completa de datos. Si los DataFrames tienen tamaños diferentes
    # DESPUÉS de la imputación, es necesario alinearlos. Asumo que se debe a NaNs.
    data_original = Hist_2016_2020[columna].dropna()
    data_simulada = data[columna].dropna()

    # Si aún fallan, es porque el número de filas en df_final y sim_df es estructuralmente diferente.
    # Si quieres comparar solo N elementos, podrías usar:
    min_len = min(len(data_original), len(data_simulada))
    data_original = data_original.iloc[:min_len]
    data_simulada = data_simulada.iloc[:min_len]

    # A. Test de Kolmogorov-Smirnov (KS)
    # Se aplica directamente a los datos continuos y limpios.
    ks_stat, ks_p_value = kstest(data_simulada, data_original)
    numéricas_df.loc[i, "KSComplemnt"] = 1 - ks_stat

    # B. Discretización (Binning) para JSD y Hellinger
    # Convertimos a distribuciones de frecuencia, usando los datos limpios.
    original_bins = pd.cut(data_original, bins=bins, include_lowest=True, duplicates='drop')
    imputada_bins = pd.cut(data_simulada, bins=bins, include_lowest=True, duplicates='drop')

    original_counts = original_bins.value_counts()
    imputada_counts = imputada_bins.value_counts()

    # Alineación
    imputada_counts, original_counts = imputada_counts.align(original_counts, fill_value=0)

    # Normalización para obtener las probabilidades (p y q)
    p = original_counts / original_counts.sum()
    q = imputada_counts / imputada_counts.sum()

    # C. Distancia de Jensen-Shannon (JSD)
    js = jensenshannon(p, q)
    numéricas_df.loc[i, "Jensen_Shannon"] = js

    # D. Distancia de Hellinger (HD)
    hellinger = np.sqrt(0.5 * ((np.sqrt(p) - np.sqrt(q))**2).sum())
    numéricas_df.loc[i, "Hellinger"] = hellinger

numérica5 = numéricas_df
numérica5

,Variable,KSComplemnt,Jensen_Shannon,Hellinger
0,PUNTAJE LECTURA CRÍTICA,0.841154,0.185203,0.197861
1,PUNTAJE COMUNICACIÓN ESCRITA,0.847627,0.137891,0.147998
2,PUNTAJE RAZONAMIENTO CUANTITATIVO,0.898913,0.048049,0.048171
3,PUNTAJE COMPETENCIAS CIUDADANAS,0.734952,0.090947,0.091064
4,PUNTAJE INGLÉS,0.762006,0.146280,0.146668


In [ ]:
numérica5.to_latex(index=False)

'\\begin{tabular}{lrrr}\n\\toprule\nVariable & KSComplemnt & Jensen_Shannon & Hellinger \\\\\n\\midrule\nPUNTAJE LECTURA CRÍTICA & 0.841154 & 0.185203 & 0.197861 \\\\\nPUNTAJE COMUNICACIÓN ESCRITA & 0.847627 & 0.137891 & 0.147998 \\\\\nPUNTAJE RAZONAMIENTO CUANTITATIVO & 0.898913 & 0.048049 & 0.048171 \\\\\nPUNTAJE COMPETENCIAS CIUDADANAS & 0.734952 & 0.090947 & 0.091064 \\\\\nPUNTAJE INGLÉS & 0.762006 & 0.146280 & 0.146668 \\\\\n\\bottomrule\n\\end{tabular}\n'

In [ ]:
#fast-MD
import numpy as np
import pandas as pd

np.random.seed(42)
# Función FAST-MD extendida para categóricas
def generate_synthetic_mixed(df, n_samples):
    synthetic = pd.DataFrame()
    for col in Hist_2016_2020.columns:
        if df[col].dtype == 'object' or Hist_2016_2020[col].dtype.name == 'category':
            # Muestreo de categorías según distribución de frecuencias
            freqs = df[col].value_counts(normalize=True)
            synthetic[col] = np.random.choice(freqs.index, size=n_samples, p=freqs.values)
        else:
            # Numéricas: normal aproximada
            mu, sigma = df[col].mean(), df[col].std()
            synthetic[col] = np.random.normal(mu, sigma, n_samples)
    return synthetic

synthetic_dp= generate_synthetic_mixed(Hist_2016_2020, n_samples=1000)
def max_discrepancy(real, synth):
    max_disc = 0
    for col in real.columns:
        stat, _ = ks_2samp(real[col], synth[col])
        max_disc = max(max_disc, stat)
    return max_disc

md_score = max_discrepancy(Hist_2016_2020, synthetic_dp)
print(f"Maximum Discrepancy entre real y sintético: {md_score:.4f}")
"----------------------------------------"
#Propensión
# ---- 1. Construir dataframe combinado ----
Hist_2016_2020['Real']=[1]*len(Hist_2016_2020['GRUPO REFERENCIA'])
synthetic_dp['Real']=[0]*len(synthetic_dp['GRUPO REFERENCIA'])
dk=pd.concat([Hist_2016_2020,synthetic_dp],ignore_index=True)
dk

# ---- 2. One-hot encode la variable categórica ----
enc = OneHotEncoder(sparse_output=False)
X = enc.fit_transform(dk[[
       'GRUPO REFERENCIA', 'NOMBRE DEL PROGRAMA ACADEMICO',
       'DEPARTAMENTO_INSTITUCION', 'MUNICIPIO_INSTITUCION',
       'NOMBRE DE LA INSTITUCION', 'COMUNICACIÓN ESCRITA',
       'COMUNICACIÓN ESCRITA_2', 'RAZONAMIENTO CUANTITATIVO',
       'RAZONAMIENTO CUANTITATIVO_2', 'LECTURA CRÍTICA', 'LECTURA CRÍTICA_2',
       'COMPETENCIAS CIUDADANAS', 'COMPETENCIAS CIUDADANAS_2', 'INGLÉS',
       'INGLÉS_2', 'año', 'PERCENTIL COMUNICACIÓN ESCRITA',
       'PUNTAJE COMUNICACIÓN ESCRITA', 'PERCENTIL RAZONAMIENTO CUANTITATIVO',
       'PUNTAJE RAZONAMIENTO CUANTITATIVO', 'PERCENTIL LECTURA CRÍTICA',
       'PUNTAJE LECTURA CRÍTICA', 'PERCENTIL COMPETENCIAS CIUDADANAS',
       'PUNTAJE COMPETENCIAS CIUDADANAS', 'PERCENTIL INGLÉS', 'PUNTAJE INGLÉS',
       'PERCENTIL COMPETENCIAS CIUDAD', 'PUNTAJE COMPETENCIAS CIUDADAN',
       'PERCENTIL COMUNICACIÓN ESCRIT', 'PERCENTIL RAZONAMIENTO CUANTI',
       'PUNTAJE RAZONAMIENTO CUANTITA']])
y = dk['Real']

# ---- 3. Entrenar clasificador ----
clf = RandomForestClassifier(n_estimators=200, random_state=0)
clf.fit(X, y)

# ---- 4. Obtener propensión (P(real)) ----
dk['propension'] = clf.predict_proba(X)[:, 1]

# ---- 5. Ver resumen ----
print(dk.groupby('Real')['propension'].describe())

NameError: name 'Hist_2016_2020' is not defined

##Presentación

In [ ]:
!streamlit --version

Streamlit, version 1.52.1


In [1]:
%%capture
!pip install streamlit-folium
!pip install geopy
!pip install streamlit pyngrok
!pip install ngrok
!pip install streamlit
!pip install streamlit-aggrid
!pip install streamlit requests pillow

In [5]:
# @title Presentación Streamlit
streamlit_code = """
import streamlit as st
import pandas as pd
import plotly.graph_objects as go
import numpy as np

# Logo or image in the sidebar
st.sidebar.image(
    "https://revistaartefacto.usta.edu.co/images/LOGO-USTA-2021-Ng.png",
    use_container_width=False
)
# Sidebar dorada
st.markdown(
    '''
    <style>
    section[data-testid="stSidebar"] {
        background-color: #F6C026 !important;
    }
    div[data-testid="stSidebarContent"] {
        background-color: #F6C026 !important;
        color: black !important;
    }
    </style>
    ''',
    unsafe_allow_html=True
)

# ==== Barra azul fija tipo encabezado institucional ====
st.markdown(
    '''
     <style>
    .top-banner {
        position: fixed;
        top: 0;
        left: 0;
        width: 100%;
        height: 28px;
        background-color: #1C3A72;
        z-index: 9999;
    }
    .block-container {
        padding-top: 40px !important;
    }
    </style>

    <div class="top-banner"></div>
    ''',
    unsafe_allow_html=True
)
# Sidebar menu
secciones = [
    "Presentación",
    "Estado del Arte",
    "Metodología",
    "Objetivos del proyecto",
    "Marco teórico",
    "Estadística exploratoria y descriptiva",
    "Simulación de variables categóricas y entropía",
    "Simulación de variables numéricas y el trilema de los datos sintéticos",
    "Referencias",
]

st.markdown(
    "<h1 style='text-align: center;'>Datos Sintéticos: Introducción a Técnicas Generativas y Medidas de Calidad</h1>",
    unsafe_allow_html=True
)
st.markdown("<hr class='custom-blue-line'>", unsafe_allow_html=True)
seccion_seleccionada = st.sidebar.radio("Selecciona una sección", secciones)
# Secciones básicas
if seccion_seleccionada == "Presentación":
    st.markdown(
    '''
    <div style='text-align: center;'>
        <h3>Presentado por:</h3>
        <h3>Diego Andrés Cleves Leguízamo</h3>
    </div>
    ''',
    unsafe_allow_html=True
)
elif seccion_seleccionada == "Estado del Arte":
    st.header("Estado del Arte")
    st.markdown("Los elementos aplicados en este proyecto no constituyen desarrollos propios, sino que son el resultado de investigaciones previas. En reconocimiento a sus aportes, presentamos un resumen estos")
    st.markdown("Instrucciones de uso: oprima el nombre del autor o autora y se desplegará el resumen de su artículo.")
    st.markdown(
    '''
    <style>
    /* Fondo del encabezado del expander */
    div[role="button"] {
        background-color: #3399ff !important;
        color: white;
        border-radius: 5px;
        padding: 8px;
        font-weight: bold;
    }

    /* Fondo del contenido dentro del expander */
    div[data-testid="stExpanderContent"] {
        background-color: #3399ff;
        color: white;
        border-radius: 5px;
        padding: 10px;
    }
    </style>
    ''',
    unsafe_allow_html=True
)

    with st.expander("Claude E. Shannon (1948) — A Mathematical Theory of Communication"):
      st.markdown("Considerado el trabajo fundacional de la teoría de la información, este texto de 55 páginas sembró las semillas de la computación, digitalización e inteligencia artificial. Proporcionó el marco teórico de la era digital.")

    with st.expander("Ronald Rubin (1993) — Statistical Disclosure Limitation"):
      st.markdown("Inspirado en los métodos de imputación de datos faltantes, en 1978 Rubin propuso la generación de datos sintéticos, los cuales permitirían proteger la privacidad de las personas y facilitar la divulgación de la estadística.")

    with st.expander("Chong K. Liew, Uinam J. Choi y Chung J. Liew (1985) — A data distortion by probability distribution"):
      st.markdown("Ensayo fundacional de la generación de datos sintéticos en la Ciencia Computacional. En él proponen los primeros métodos y medidas de calidad de esta disciplina. Aunque menos reconocidos que Shannon y Rubin sus aportes no fueron pocos.")

    with st.expander("Joerg Drechsler y Anna-Carolina Haensch (2023) — 30 Years of Synthetic Data"):
      st.markdown("Artículo que recopila la historia de la generación de datos sintéticos de 1993 a 2023. Su valor radica en las referencias que hace de los trabajos más importantes de esta disciplina.")

    with st.expander("Cynthia Dwork y Aaron Roth (2014) — The Algorithmic Foundations of Differential Privacy"):
      st.markdown("En este libro se compila buena parte de la literatura dedicada a la privacidad diferencial, una metodología en la cual se agrega ruido a los datos que se desean enmascarar con el fin de complicar su detección.")

    with st.expander("Joshua Snoke et al (2018) — Combining propensity score methods with variational autoencoders for generating synthetic data in presence of latent sub‑groups"):
      st.markdown("Este estudio identifica primero un problema inherente a sintetización de datos: la invisibilización de subgrupos cuya probabilidad de ser seleccionados es un evento cero.")

elif seccion_seleccionada == "Metodología":
    st.markdown("<h3 style='text-align: center;'>Metodología</h3>", unsafe_allow_html=True)
    tab1,tab2 = st.tabs(["Manejo de datos faltantes", "Generación de datos sintéticos"])
    with tab1:
      diagrama_faltantes_estilizado = '''
      digraph G {
      // Configuración general del gráfico
      rankdir=TB; // Top to Bottom direction
      splines=polyline; // Use straight lines with elbow bends for neater layout
      node [fontname="Helvetica", fontsize=10, penwidth=1.5]; // Default font and border thickness
      edge [fontname="Helvetica", fontsize=9, fontcolor="#555555"]; // Default edge style

      // =====================================
      // Estilos y Definición de Nodos
      // =====================================

      // --- ESTILO 1: Nodos de Inicio, Fin y Convergencia (Morado, Redondeados) ---
      node [shape=Mrecord, style="filled", fillcolor="#D8BFD8", color="#4B0082"];
      n1 [label="Identificar capa o nivel de los datos"];
      cont [label="Continuar con la siguiente etapa"];
      b2 [label="Finalizar el proceso"];

      // --- ESTILO 2: Nodos de Proceso/Actividad (Verde, Rectangulares) ---
      node [shape=box, style="filled", fillcolor="#E6F2E6", color="#006400"];
      n2 [label="Detectar valores faltantes"];
      a1 [label="Identificar características estadísticas de los datos incompletos"];
      a2 [label="Imputar los datos"];
      b1 [label="Filtrar datos útiles para un nuevo planteamiento"];

      // --- ESTILO 3: Nodos de Decisión (Beige, Rombos) ---
      // Aumentamos un poco el 'height' para que el texto quepa bien en el rombo
      node [shape=diamond, style="filled", fillcolor="#F2E6CC", color="#8B4513", height=1.2];
      d1 [label="¿Se identificaron datos faltantes?"];
      d2 [label="¿Los datos faltantes son relevantes?"];
      d3 [label="¿Es posible imputarlos?"];
      d4 [label="¿Puede reformularse la simulación con los datos disponibles?"];


      // =====================================
      // Conexiones (Flujo Lógico)
      // =====================================

      // Flujo Inicial
      n1 -> n2 [weight=5]; // Higher weight keeps main flow straight
      n2 -> d1 [weight=5];

      // --- Ramas de Decisión ---

      // Decisión 1: ¿Hay faltantes?
      d1:s -> d2 [label="Sí", color="#006400", fontcolor="#006400", weight=5]; // Camino principal "Sí" en verde
      d1:w -> cont:w [label="No", color="#8B0000", fontcolor="#8B0000"]; // Camino "No" en rojo, conectando por la izquierda

      // Decisión 2: ¿Son relevantes?
      d2:s -> d3 [label="Sí", color="#006400", fontcolor="#006400", weight=5];
      d2:w -> cont:w [label="No", color="#8B0000", fontcolor="#8B0000"];

      // Decisión 3: ¿Imputables?
      // Camino del SÍ (Imputación)
      d3:e -> a1 [label="Sí", color="#006400", fontcolor="#006400"];
      a1 -> a2;
      a2 -> cont:e; // Conecta por la derecha para balancear el gráfico

      // Camino del NO (No imputable)
      d3:s -> d4 [label="No", color="#8B0000", fontcolor="#8B0000", weight=5];

      // Decisión 4: ¿Reformular?
      // Camino del SÍ (Reformulación)
      d4:e -> b1 [label="Sí", color="#006400", fontcolor="#006400"];
      b1 -> cont:e;

      // Camino del NO (Finalizar)
      d4:s -> b2 [label="No", color="#8B0000", fontcolor="#8B0000"];
      }
      '''
      st.graphviz_chart(diagrama_faltantes_estilizado, use_container_width=True)

    with tab2:
      st.markdown('''
        1. **Analizar riesgos de privacidad**
        🔹 Identificar posibles brechas que expongan información sensible.

        2. **Verificar proporción de datos reales ≤ 20%**
        🔹 Asegurar que los datos sintéticos no contengan demasiada información real.

        3. **Evaluar similitud y características estadísticas**
        🔹 Comparar registros generados con los reales para mantener realismo sin comprometer privacidad.

        4. **Calcular propensión y validar que no supere el 70%**
        🔹 Evaluar si los datos sintéticos cumplen criterios de similitud y utilidad.

        5. **Comprobar utilidad para entrenamiento**
        🔹 Determinar si los datos generados son adecuados para entrenar modelos.

        6. **Si alguna condición falla → replantear simulación**
        🔹 Ajustar parámetros, mejorar similitud y reducir exposición de datos reales.

        7. **Si todo es satisfactorio → aprobar los datos**

''')


elif seccion_seleccionada == "Objetivos del proyecto":
    st.header("Objetivo General")
    st.markdown("Contrastar las técnicas de generación de datos sintéticos.")
    st.header("Objetivos Específicos")
    objetivos = [
        "Evaluar los datos sintéticos generados de acuerdo a las medidas de calidad investigadas y técnicas generativas usadas.",
        "Por medio de reducción al absurdo, mostrar errores que se cometen al generar sintéticos.",
        "Analizar las  medidas de calidad investigadas.",
        "Analizar e implementar las técnicas generativas investigadas, consideradas pertinentes.",


    ]
    for obj in objetivos:
        st.markdown(f"- {obj}")
elif seccion_seleccionada == "Marco teórico":
    st.markdown("<h3 style='text-align: center;'>Marco teórico</h3>", unsafe_allow_html=True)
    tab1,tab2,tab3,tab4= st.tabs(["Datos sintéticos ¿qué son?","Antecedentes","Industrias donde son usados","Teoremas y leyes importantes"])
    with tab1:
     with st.expander("Definición"):
      st.markdown("Son datos simulados que replican las características estadísticas de datos reales tales como su distribución empírica, distribución marginal, media, moda, varianza, etc.")
     with st.expander("Tipos"):
      st.markdown("Los datos completamente sintéticos son aquellos que en su totalidad fueron simulados.")
      st.markdown("Los datos parcialmente sintéticos son aquellos que combinan datos provenientes de simulaciones con datos reales.")
     with st.expander("Ejemplos"):
      st.markdown("Se han podido replicar datos tabulares, imágenes, series de tiempo,...")
     with st.expander("Métodos de generación"):
      data = {
      'Tipo de Modelo': [
        'Modelos Basados en Aprendizaje Profundo',
        'Modelos Basados en Árboles',
        'Modelos Estadísticos',
        'Modelos Basados en Reglas'],
      'Descripción': [
        'Redes Neuronales como GANs o VAEs aprenden las distribuciones de datos complejos (imágenes, series temporales).',
        'Random Forests para generar datos categóricos.',
        'Regresión Lineal o Múltiple para generar datos basados en parámetros estadísticos (medias, covarianzas).',
        'Generación basada en reglas predefinidas por expertos en el dominio.'],
      'Ventajas': [
        'Alta fidelidad en la distribución y relaciones complejas.',
        'Relativamente simple, rápido y funciona bien para datos tabulares (estructurados).',
        'Rápidos y fáciles de interpretar.',
        'Garantiza la plausibilidad física o lógica.'],
      'Inconvenientes': [
        'Requieren grandes conjuntos de datos de entrenamiento y alta potencia computacional.',
        'Puede fallar al capturar relaciones no lineales complejas.',
        'Tienen baja fidelidad en *datasets* no Gaussianos o con relaciones complejas.',
        'Puede ser subjetivo y no capturar relaciones estadísticas inesperadas.']}
      df_metodos = pd.DataFrame(data)
      st.markdown("Métodos Comunes para la Generación de Datos Sintéticos")
      st.caption("Resumen de las técnicas más utilizadas, sus fortalezas y debilidades.")
      st.dataframe(df_metodos, use_container_width=True,hide_index=True)
    with tab2:
      with st.expander("Antecedentes en medicina"):
        st.image("https://cbaglobal.com.ar/wp-content/uploads/2024/06/escuadro-731.jpg",use_container_width=False)
        st.image("https://www.mcgill.ca/oss/files/oss/styles/hd/public/tuskegee_study.jpg?itok=QaQOKZlx&timestamp=1548275432",use_container_width=False)
      with st.expander("Antecedentes políticos"):
        st.image("https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTA60jNQ8axIwmKGqeJJZiwGHq9lT6z7o4hVA&s",use_container_width=False)
    with tab3:
      with st.expander("Usados en:"):
        objetivos = [
        "Sanidad.",
        "Ciberseguridad.",
        "Inteligencia artificial.",
        "Robótica."]
        for obj in objetivos:
          st.markdown(f"- {obj}")
      with st.expander("Casos de éxito"):
        st.image("https://i.ytimg.com/vi/C8SQE5EdflY/hq720.jpg?sqp=-oaymwEhCK4FEIIDSFryq4qpAxMIARUAAAAAGAElAADIQj0AgKJD&rs=AOn4CLACrn2scPkBOljsc0GGpMaKz0LShw",use_container_width=False)
        st.image("https://gomoot.com/wp-content/uploads/2025/08/deepseek3-1.webp",use_container_width=False)
        st.image("https://blog.cordatus.ai/wp-content/uploads/2025/07/phi4.webp",use_container_width=False)
    with tab4:
      with st.expander("Teorema de Sklar"):
        st.markdown("cualquier función de distribución conjunta multivariada puede ser escrita en términos de distribuciones marginales univariadas y una cópula, que describe la estructura de dependencia entre las variables.")
        formula_sklar = r'''H(x_1, x_2, \dots, x_n) = C(F_1(x_1), F_2(x_2), \dots, F_n(x_n))'''
        st.latex(formula_sklar)
      with st.expander("Ley fundamental de la recuperación de la información"):
        st.markdown("También conocidos como ataques de reconstrucción(reconstruction attacks, en inglés), afirma que demasiadas respuestas correctas a demasiadas preguntas destruyen la privacidad.")
      with st.expander("Pérdida de información"):
        st.markdown("Postulado por Rubin en 1993 y por Shannon en 1948, las categorías individuales y las relaciones menos probables tienden a desaparecer.")

elif seccion_seleccionada == "Estadística exploratoria y descriptiva":
    st.header("Estadística descriptiva y exploratoria")
    tab1,tab2,tab3= st.tabs(["Estructura de los datos","Datos faltantes por variable","Muestra seleccionada"])
    with tab2:
      data = {
        'Tipo': ['objeto', 'objeto', 'objeto', 'objeto', 'objeto', 'objeto',
             'int64', 'int64', 'int64', 'int64',
             'float64', 'float64', 'float64', 'float64',
             'float64', 'float64'],
      2016: [0, 0, 0, 0, 0, 0, 0, 0, 331, 331, 392, 392, 291, 291, 364, 364],
      2017: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 21, 21, 14, 14, 19, 19],
      2018: [0, 0, 0, 0, 0, 0, 81, 81, 0, 0, 103, 103, 132, 132, 82, 82],
      2019: [0, 0, 0, 0, 0, 0, 81, 81, 0, 0, 127, 127, 92, 92, 101, 101],
      2020: [0, 0, 0, 0, 0, 0, 216, 216, 0, 0, 187, 187, 252, 252, 183, 183]}
      index_labels = [
        'Apellidos Y Nombres', 'Grupo Referencia', 'Nombre Del Programa Académico',
        'Departamento institución', 'Municipio institución', 'Nombre De La Institución',
        'Percentil Comunicación Escrita', 'Puntaje Comunicación Escrita',
        'Percentil Razonamiento Cuantitativo', 'Puntaje Razonamiento Cuantitativo',
        'Percentil Lectura Crítica', 'Puntaje Lectura Crítica',
        'Percentil Competencias Ciudadanas', 'Puntaje Competencias Ciudadanas',
        'Percentil Inglés', 'Puntaje Inglés']
      df = pd.DataFrame(data, index=index_labels)
      df.index.name = 'Variable'
      df.columns.name = 'Año'
      st.markdown("Tipos de variables y valores faltantes por año: 2016-2020")
      st.caption("Tabla 1: Conteo de valores faltantes (NaN) por variable y por año en el conjunto de datos.")
      st.dataframe(df)
    with tab1:
      data = {
      'Año': [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024],
      'Observaciones': [11033, 10763, 10166, 10691, 12712, 14222, 11212, 10986, 13007],
      'Covariables': [16, 16, 16, 16, 16, 17, 17, 17, 17],
      'Datos': [176528, 172208, 162656, 171056, 203392, 241774, 190604, 186762, 221119],
      'Datos faltantes (%)': ['1.56', '0.06', '0.49', '0.47', '0.82', '0.00', '0.00', '0.00', '0.00'],
      '¿Imputar?': ['Sí', 'Sí', 'Sí', 'Sí', 'Sí', 'No', 'No', 'No', 'No']}

      df_resumen = pd.DataFrame(data)

      df_resumen['Datos faltantes (%)'] = df_resumen['Datos faltantes (%)'].str.replace(',', '.')


      df_resumen = df_resumen.set_index('Año')
      df_resumen.columns.name = None # Limpia el nombre de la columna superior
      st.markdown("Resumen de Datos Faltantes por Año (2016-2024)")
      st.caption("Tabla 2: Conteo de observaciones, covariables y porcentaje de datos faltantes en el conjunto total.")
      st.dataframe(df_resumen)
    with tab3:
      data = {
      'Año': [2016, 2017, 2018, 2019, 2020],
      'Evaluados': [245162, 248855, 240403, 269431, 253114],
      'Mejores': [11033, 10763, 10166, 10691, 12712],
      '% mejores': [4.5003, 4.3250, 4.2287, 3.9679, 5.0222]}
      # Crear el DataFrame
      df_estudiantes = pd.DataFrame(data)

      # Formatear el porcentaje para la visualización (opcional, pero mejora la lectura)
      df_estudiantes['% mejores'] = df_estudiantes['% mejores'].map('{:.4f}%'.format)

      # Establecer el 'Año' como índice (opcional)
      df_estudiantes = df_estudiantes.set_index('Año')
      df_estudiantes.columns.name = None

      # --- 2. Renderizar en Streamlit ---

      # Título de la tabla
      st.markdown("Estudiantes Evaluados (2016-2020)")
      st.caption("Tabla 3: estudiantes seleccionados como los 'Mejores' por el ICFES.")

      # Crear el DataFrame
      df_estudiantes = pd.DataFrame(data)

      # Formatear el porcentaje para la visualización (opcional, pero mejora la lectura)
      df_estudiantes['% mejores'] = df_estudiantes['% mejores'].map('{:.4f}%'.format)

      # Establecer el 'Año' como índice (opcional)
      df_estudiantes = df_estudiantes.set_index('Año')
      df_estudiantes.columns.name = None
      st.dataframe(df_estudiantes)

elif seccion_seleccionada == "Simulación de variables categóricas y entropía":
  tab1,tab2,tab3= st.tabs(["Simulación de variables categóricas","Medidas de calidad","Concepto de entropía"])
  with tab1:
    st.header("Simulación de variables categóricas")
    st.subheader("¿Cómo se simuló?")
    st.markdown("Siguiendo el teorema de Sklar, se capturó la estructura de dependencia entre variables. Para ello se realizó la simulación por medio de un árbol de probabilidad condicional supervisado.")
    url = f"https://docs.google.com/spreadsheets/d/18TbHDHJcjv9HZug3jEiW8RLio_RUdaCd/export?format=csv&gid=1388718316"
    st.dataframe(pd.read_csv(url),hide_index=True)
  with tab2:
    with st.expander("Medidas de similitud"):
     data = {
        'Variable': [
        'Grupo de Referencia',
        'Nombre del Programa Académico',
        'Departamento de la Institución',
        'Municipio de la Institución',
        'Nombre de la Institución'],
      'TCP (%)': ['12.6%', '2.88%', '17.7%', '21.31%', '3.03%'],
      'Chi2': [0.973, 1.000, 0.9995, 1.000, 1.000],
      'JS': [0.0071, 0.0412, 0.0099, 0.0128, 0.0267],
      'H': [0.0071, 0.0431, 0.0102, 0.0132, 0.0274]}
     df_calidad = pd.DataFrame(data)
     formatters = {
        'Chi2': '{:.4f}'.format,
        'JS': '{:.4f}'.format,
        'H': '{:.4f}'.format}
     st.caption("Tabla de calidad de las variables categóricas, donde se evalúan métricas de fidelidad.")
     st.dataframe(df_calidad.style.format(formatters), use_container_width=True, hide_index=True)
    with st.expander("Privacidad"):
      data = {
        'count': [55365.0, 55365.0],
        'mean': [0.478077, 0.521623],
        'std': [0.103485, 0.095807],
        'min': [0.000000, 0.115183],
        '25%': [0.448742, 0.473766],
        '50%': [0.492007, 0.506168],
        '75%': [0.526161, 0.550003],
        'max': [0.951634, 1.000000]}

      index_labels = [0, 1]  # Los valores de la columna 'Real'
      df_descripcion = pd.DataFrame(data, index=index_labels)
      df_descripcion.index.name = 'Real'
      st.dataframe(df_descripcion.style.format(precision=6), use_container_width=True)
  with tab3:
      st.header("Concepto de entropía")
      st.markdown("Medida de incertidubre, la cual puede ser mayor o menor en función de los estados posibles que esa variable pueda tomar.")
      with st.expander("Manifestación"):
       data = {
        'Variable': ['Depto. institución', 'Municipio institución', 'Institución', 'Programa académico', 'Grupo de referencia'],
        'Cat. Sintéticas': [27, 70, 252, 666, 23],
        'Cat. Reales': [27, 80, 266, 710, 23],
        'Cat. Perdidas': [0, 10, 14, 44, 0],
        'Moda (Tabla 1)': ['Bogotá', 'Bogotá, D. C.', 'Los Andes', 'Derecho', 'Ingeniería'],
        'Moda (Tabla 2)': ['Bogotá', 'Bogotá, D. C.', 'Los Andes', 'Derecho', 'Ingeniería']}
       df_comparativo = pd.DataFrame(data)
       st.dataframe(df_comparativo, use_container_width=True, hide_index=True)


elif seccion_seleccionada == "Simulación de variables numéricas y el trilema de los datos sintéticos":
  tab1,tab2,tab3,tab4= st.tabs(["Simulación de variables categóricas","Medidas de calidad","Trilema de los datos sintéticos","Utilidad"])
  with tab1:
    st.header("Simulación de variables numérica")
    st.subheader("¿Cómo se simuló?")
    st.markdown("Siguiendo el teorema de Sklar, se intentó capturar la estructura de dependencia entre variables. Se recurrió a cópulas gaussianas, cuyo proceso de construcción se automatizó con la librería SDV del MIT.")
    url = f"https://docs.google.com/spreadsheets/d/1bNg1I1oo0FKlLKUbjoS3XpY7Y-PJwIiJ/export?format=csv&gid=1261021413"
    st.dataframe(pd.read_csv(url),hide_index=True)
  with tab2:
    data = {
        'Variable': [
        'Grupo de Referencia',
        'Nombre del Programa Académico',
        'Departamento de la Institución',
        'Municipio de la Institución',
        'Nombre de la Institución'],
      'KS complemento': [0.9127, 0.9595, 0.9102, 0.9724, 0.9579],
      'JS': [0.0916, 0.0463, 0.1288, 0.0133, 0.1065],
      'H': [0.1002, 0.0466, 0.1316, 0.0135, 0.1181]}
    df_calidad = pd.DataFrame(data)
    formatters = {
        'Chi2': '{:.4f}'.format,
        'JS': '{:.4f}'.format,
        'H': '{:.4f}'.format}
    st.caption("Tabla de calidad de las variables categóricas, donde se evalúan métricas de fidelidad.")
    st.dataframe(df_calidad.style.format(formatters), use_container_width=True, hide_index=True)
  with tab3:
    with st.expander("Trilema de los datos sintéticos"):
      st.markdown("Flexibilidad, utilidad y privacidad son los pilares de la generación de datos sintéticos, pero producto de la ley fundamental de recuperación de información se deben sacrificar dos de las tres para que la propuesta funcione." )
  with tab4:

    resultados = pd.DataFrame({
      "Modelo": ["Referencia (Real)", "Prueba (Sintético)"],
      "MAE": [0.6754, 1.5698],
      "RMSE": [1.0635, 3.5409],
      "R2": [0.9927, 0.9195]})
    st.table(resultados)

    st.markdown('''
    - El modelo con datos reales alcanza un **R2 muy alto (0.993)**, indicando excelente ajuste.
    - El modelo con datos sintéticos tiene un **R2 de 0.92**, mostrando que conserva bastante la información pero con mayor error (MAE y RMSE más altos).
    ''')


elif seccion_seleccionada == "Referencias":
    st.header("Referencias")

    st.markdown('''
1. [Evaluating Synthetic Data: The Million-Dollar Question](https://towardsdatascience.com/evaluating-synthetic-data-the-million-dollar-question-a54701d1b621/)
2. [ScienceDirect: Evaluating synthetic data quality](https://www.sciencedirect.com/science/article/pii/S1386505624000765)
3. [Bluegen: How do I know that the synthetic data is of the right quality for my use case?](https://bluegen.ai/how-do-i-know-that-the-synthetic-data-is-of-the-right-quality-for-my-use-case/)
4. [UN News: Synthetic data and statistics](https://news.un.org/es/story/2018/11/1446671)
5. [UN: Universal Declaration of Human Rights](https://www.un.org/es/about-us/universal-declaration-of-human-rights#:~:text=Art%C3%ADculo%2012,contra%20tales%20injerencias%20o%20ataques.)
6. [ACL Anthology 2025](https://aclanthology.org/2025.cl-1.6.pdf)
7. [PMC Article on Synthetic Data](https://pmc.ncbi.nlm.nih.gov/articles/PMC9951365/)
8. [ScienceDirect Article](https://www.sciencedirect.com/science/article/pii/S2001037024002393)
9. [SCB: Statistical Analysis of Masked Data](https://www.scb.se/contentassets/ca21efb41fee47d293bbee5bf7be7fb3/statistical-analysis-of-masked-data.pdf)
10. [US Census Technical Paper](https://www2.census.gov/ces/tp/tp-2003-10.pdf)
11. [Royal Society: Synthetic Data Survey](https://royalsociety.org/-/media/policy/projects/privacy-enhancing-technologies/Synthetic_Data_Survey-24.pdf)
12. [Policy Review: Politics of Synthetic Data](https://policyreview.info/articles/news/politics-of-synthetic-data-performance-metrics/1761)
13. [UNESCO AI Ethics Recommendation](https://www.unesco.org/en/artificial-intelligence/recommendation-ethics)
14. [IBM AI Ethics](https://www.ibm.com/think/topics/ai-ethics#:~:text=Examples%20of%20AI%20ethics%20issues,%2C%20trust%2C%20and%20technology%20misuse.)
15. [NVIDIA Glossary: Synthetic Data Generation](https://www.nvidia.com/en-us/glossary/synthetic-data-generation/)
16. [DataCamp Tutorial: Synthetic Data Generation](https://www.datacamp.com/tutorial/synthetic-data-generation)
17. [SDV: Synthetic Data Vault](https://sdv.dev/)
''', unsafe_allow_html=True)
"""
# Guardar el código Streamlit en un archivo
with open("/content/streamlit_plotly_timeseries.py", "w") as f:
    f.write(streamlit_code)

# Establecer el token de autenticación de ngrok
from pyngrok import ngrok
ngrok.set_auth_token("2tzdV3oofirn68JeCgolauxDQUt_75mTMJ3ko2AKPPnGaxi9A")

# Crear el túnel de ngrok a Streamlit en el puerto 8501
public_url = ngrok.connect(8501)
print(f"Streamlit app is live at: {public_url}")

# Ejecutar la aplicación Streamlit en segundo plano
!streamlit run /content/streamlit_plotly_timeseries.py &

<>:301: SyntaxWarning: invalid escape sequence '\d'
<>:301: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_5102/909307958.py:301: SyntaxWarning: invalid escape sequence '\d'
  formula_sklar = r'''H(x_1, x_2, \dots, x_n) = C(F_1(x_1), F_2(x_2), \dots, F_n(x_n))'''


Streamlit app is live at: NgrokTunnel: "https://07cd-34-125-93-7.ngrok-free.app" -> "http://localhost:8501"



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.93.7:8501

2026-04-20 08:15:57.955 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.
  Stopping...


In [ ]:
# @title Presentación Streamlit(back-up)
streamlit_code = """
import streamlit as st
import pandas as pd
import plotly.graph_objects as go
import numpy as np

# Logo or image in the sidebar
st.sidebar.image(
    "https://revistaartefacto.usta.edu.co/images/LOGO-USTA-2021-Ng.png",
    use_container_width=False
)
# Sidebar dorada
st.markdown(
    '''
    <style>
    section[data-testid="stSidebar"] {
        background-color: #F6C026 !important;
    }
    div[data-testid="stSidebarContent"] {
        background-color: #F6C026 !important;
        color: black !important;
    }
    </style>
    ''',
    unsafe_allow_html=True
)

# ==== Barra azul fija tipo encabezado institucional ====
st.markdown(
    '''
     <style>
    .top-banner {
        position: fixed;
        top: 0;
        left: 0;
        width: 100%;
        height: 28px;
        background-color: #1C3A72;
        z-index: 9999;
    }
    .block-container {
        padding-top: 40px !important;
    }
    </style>

    <div class="top-banner"></div>
    ''',
    unsafe_allow_html=True
)
# Sidebar menu
secciones = [
    "Presentación",
    "Estado del Arte",
    "Metodología",
    "Objetivos del proyecto",
    "Marco teórico",
    "Estadística exploratoria y descriptiva",
    "Simulación de variables categóricas y entropía",
    "Simulación de variables numéricas y el trilema de los datos sintéticos",
    "Referencias",
]

st.markdown(
    "<h1 style='text-align: center;'>Datos Sintéticos: Introducción a Técnicas Generativas y Medidas de Calidad</h1>",
    unsafe_allow_html=True
)
st.markdown("<hr class='custom-blue-line'>", unsafe_allow_html=True)
seccion_seleccionada = st.sidebar.radio("Selecciona una sección", secciones)
# Secciones básicas
if seccion_seleccionada == "Presentación":
    st.markdown(
    '''
    <div style='text-align: center;'>
        <h3>Presentado por:</h3>
        <h3>Diego Andrés Cleves Leguízamo</h3>
    </div>
    ''',
    unsafe_allow_html=True
)
elif seccion_seleccionada == "Estado del Arte":
    st.header("Estado del Arte")
    st.markdown("Los elementos aplicados en este proyecto no constituyen desarrollos propios, sino que son el resultado de investigaciones previas. En reconocimiento a sus aportes, presentamos un resumen estos")
    st.markdown("Instrucciones de uso: oprima el nombre del autor o autora y se desplegará el resumen de su artículo.")
    st.markdown(
    '''
    <style>
    /* Fondo del encabezado del expander */
    div[role="button"] {
        background-color: #3399ff !important;
        color: white;
        border-radius: 5px;
        padding: 8px;
        font-weight: bold;
    }

    /* Fondo del contenido dentro del expander */
    div[data-testid="stExpanderContent"] {
        background-color: #3399ff;
        color: white;
        border-radius: 5px;
        padding: 10px;
    }
    </style>
    ''',
    unsafe_allow_html=True
)

    with st.expander("Claude E. Shannon (1948) — A Mathematical Theory of Communication"):
      st.markdown("Considerado el trabajo fundacional de la teoría de la información, este texto de 55 páginas sembró las semillas de la computación, digitalización e inteligencia artificial. Proporcionó el marco teórico de la era digital.")

    with st.expander("Ronald Rubin (1993) — Statistical Disclosure Limitation"):
      st.markdown("Inspirado en los métodos de imputación de datos faltantes, en 1978 Rubin propuso la generación de datos sintéticos, los cuales permitirían proteger la privacidad de las personas y facilitar la divulgación de la estadística.")

    with st.expander("Chong K. Liew, Uinam J. Choi y Chung J. Liew (1985) — A data distortion by probability distribution"):
      st.markdown("Ensayo fundacional de la generación de datos sintéticos en la Ciencia Computacional. En él proponen los primeros métodos y medidas de calidad de esta disciplina. Aunque menos reconocidos que Shannon y Rubin sus aportes no fueron pocos.")

    with st.expander("Joerg Drechsler y Anna-Carolina Haensch (2023) — 30 Years of Synthetic Data"):
      st.markdown("Artículo que recopila la historia de la generación de datos sintéticos de 1993 a 2023. Su valor radica en las referencias que hace de los trabajos más importantes de esta disciplina.")

    with st.expander("Cynthia Dwork y Aaron Roth (2014) — The Algorithmic Foundations of Differential Privacy"):
      st.markdown("En este libro se compila buena parte de la literatura dedicada a la privacidad diferencial, una metodología en la cual se agrega ruido a los datos que se desean enmascarar con el fin de complicar su detección.")

    with st.expander("Joshua Snoke et al (2018) — Combining propensity score methods with variational autoencoders for generating synthetic data in presence of latent sub‑groups"):
      st.markdown("Este estudio identifica primero un problema inherente a sintetización de datos: la invisibilización de subgrupos cuya probabilidad de ser seleccionados es un evento cero.")

elif seccion_seleccionada == "Metodología":
    st.markdown("<h3 style='text-align: center;'>Metodología</h3>", unsafe_allow_html=True)
    tab1,tab2 = st.tabs(["Manejo de datos faltantes", "Generación de datos sintéticos"])
    with tab1:
      diagrama_faltantes_estilizado = '''
      digraph G {
      // Configuración general del gráfico
      rankdir=TB; // Top to Bottom direction
      splines=polyline; // Use straight lines with elbow bends for neater layout
      node [fontname="Helvetica", fontsize=10, penwidth=1.5]; // Default font and border thickness
      edge [fontname="Helvetica", fontsize=9, fontcolor="#555555"]; // Default edge style

      // =====================================
      // Estilos y Definición de Nodos
      // =====================================

      // --- ESTILO 1: Nodos de Inicio, Fin y Convergencia (Morado, Redondeados) ---
      node [shape=Mrecord, style="filled", fillcolor="#D8BFD8", color="#4B0082"];
      n1 [label="Identificar capa o nivel de los datos"];
      cont [label="Continuar con la siguiente etapa"];
      b2 [label="Finalizar el proceso"];

      // --- ESTILO 2: Nodos de Proceso/Actividad (Verde, Rectangulares) ---
      node [shape=box, style="filled", fillcolor="#E6F2E6", color="#006400"];
      n2 [label="Detectar valores faltantes"];
      a1 [label="Identificar características estadísticas de los datos incompletos"];
      a2 [label="Imputar los datos"];
      b1 [label="Filtrar datos útiles para un nuevo planteamiento"];

      // --- ESTILO 3: Nodos de Decisión (Beige, Rombos) ---
      // Aumentamos un poco el 'height' para que el texto quepa bien en el rombo
      node [shape=diamond, style="filled", fillcolor="#F2E6CC", color="#8B4513", height=1.2];
      d1 [label="¿Se identificaron datos faltantes?"];
      d2 [label="¿Los datos faltantes son relevantes?"];
      d3 [label="¿Es posible imputarlos?"];
      d4 [label="¿Puede reformularse la simulación con los datos disponibles?"];


      // =====================================
      // Conexiones (Flujo Lógico)
      // =====================================

      // Flujo Inicial
      n1 -> n2 [weight=5]; // Higher weight keeps main flow straight
      n2 -> d1 [weight=5];

      // --- Ramas de Decisión ---

      // Decisión 1: ¿Hay faltantes?
      d1:s -> d2 [label="Sí", color="#006400", fontcolor="#006400", weight=5]; // Camino principal "Sí" en verde
      d1:w -> cont:w [label="No", color="#8B0000", fontcolor="#8B0000"]; // Camino "No" en rojo, conectando por la izquierda

      // Decisión 2: ¿Son relevantes?
      d2:s -> d3 [label="Sí", color="#006400", fontcolor="#006400", weight=5];
      d2:w -> cont:w [label="No", color="#8B0000", fontcolor="#8B0000"];

      // Decisión 3: ¿Imputables?
      // Camino del SÍ (Imputación)
      d3:e -> a1 [label="Sí", color="#006400", fontcolor="#006400"];
      a1 -> a2;
      a2 -> cont:e; // Conecta por la derecha para balancear el gráfico

      // Camino del NO (No imputable)
      d3:s -> d4 [label="No", color="#8B0000", fontcolor="#8B0000", weight=5];

      // Decisión 4: ¿Reformular?
      // Camino del SÍ (Reformulación)
      d4:e -> b1 [label="Sí", color="#006400", fontcolor="#006400"];
      b1 -> cont:e;

      // Camino del NO (Finalizar)
      d4:s -> b2 [label="No", color="#8B0000", fontcolor="#8B0000"];
      }
      '''
      st.graphviz_chart(diagrama_faltantes_estilizado, use_container_width=True)

    with tab2:
      st.markdown('''
        1. **Analizar riesgos de privacidad**
        🔹 Identificar posibles brechas que expongan información sensible.

        2. **Verificar proporción de datos reales ≤ 20%**
        🔹 Asegurar que los datos sintéticos no contengan demasiada información real.

        3. **Evaluar similitud y características estadísticas**
        🔹 Comparar registros generados con los reales para mantener realismo sin comprometer privacidad.

        4. **Calcular propensión y validar que no supere el 70%**
        🔹 Evaluar si los datos sintéticos cumplen criterios de similitud y utilidad.

        5. **Comprobar utilidad para entrenamiento**
        🔹 Determinar si los datos generados son adecuados para entrenar modelos.

        6. **Si alguna condición falla → replantear simulación**
        🔹 Ajustar parámetros, mejorar similitud y reducir exposición de datos reales.

        7. **Si todo es satisfactorio → aprobar los datos**

''')


elif seccion_seleccionada == "Objetivos del proyecto":
    st.header("Objetivo General")
    st.markdown("Contrastar las técnicas de generación de datos sintéticos.")
    st.header("Objetivos Específicos")
    objetivos = [
        "Evaluar los datos sintéticos generados de acuerdo a las medidas de calidad investigadas y técnicas generativas usadas.",
        "Por medio de reducción al absurdo, mostrar errores que se cometen al generar sintéticos.",
        "Analizar las  medidas de calidad investigadas.",
        "Analizar e implementar las técnicas generativas investigadas, consideradas pertinentes.",


    ]
    for obj in objetivos:
        st.markdown(f"- {obj}")
elif seccion_seleccionada == "Marco teórico":
    st.markdown("<h3 style='text-align: center;'>Marco teórico</h3>", unsafe_allow_html=True)
    tab1,tab2,tab3,tab4= st.tabs(["Datos sintéticos ¿qué son?","Antecedentes","Industrias donde son usados","Teoremas y leyes importantes"])
    with tab1:
     with st.expander("Definición"):
      st.markdown("Son datos simulados que replican las características estadísticas de datos reales tales como su distribución empírica, distribución marginal, media, moda, varianza, etc.")
     with st.expander("Tipos"):
      st.markdown("Los datos completamente sintéticos son aquellos que en su totalidad fueron simulados.")
      st.markdown("Los datos parcialmente sintéticos son aquellos que combinan datos provenientes de simulaciones con datos reales.")
     with st.expander("Ejemplos"):
      st.markdown("Se han podido replicar datos tabulares, imágenes, series de tiempo,...")
     with st.expander("Métodos de generación"):
      data = {
      'Tipo de Modelo': [
        'Modelos Basados en Aprendizaje Profundo',
        'Modelos Basados en Árboles',
        'Modelos Estadísticos',
        'Modelos Basados en Reglas'],
      'Descripción': [
        'Redes Neuronales como GANs o VAEs aprenden las distribuciones de datos complejos (imágenes, series temporales).',
        'Random Forests para generar datos categóricos.',
        'Regresión Lineal o Múltiple para generar datos basados en parámetros estadísticos (medias, covarianzas).',
        'Generación basada en reglas predefinidas por expertos en el dominio.'],
      'Ventajas': [
        'Alta fidelidad en la distribución y relaciones complejas.',
        'Relativamente simple, rápido y funciona bien para datos tabulares (estructurados).',
        'Rápidos y fáciles de interpretar.',
        'Garantiza la plausibilidad física o lógica.'],
      'Inconvenientes': [
        'Requieren grandes conjuntos de datos de entrenamiento y alta potencia computacional.',
        'Puede fallar al capturar relaciones no lineales complejas.',
        'Tienen baja fidelidad en *datasets* no Gaussianos o con relaciones complejas.',
        'Puede ser subjetivo y no capturar relaciones estadísticas inesperadas.']}
      df_metodos = pd.DataFrame(data)
      st.markdown("Métodos Comunes para la Generación de Datos Sintéticos")
      st.caption("Resumen de las técnicas más utilizadas, sus fortalezas y debilidades.")
      st.dataframe(df_metodos, use_container_width=True,hide_index=True)
    with tab2:
      with st.expander("Antecedentes en medicina"):
        st.image("https://cbaglobal.com.ar/wp-content/uploads/2024/06/escuadro-731.jpg",use_container_width=False)
        st.image("https://www.mcgill.ca/oss/files/oss/styles/hd/public/tuskegee_study.jpg?itok=QaQOKZlx&timestamp=1548275432",use_container_width=False)
      with st.expander("Antecedentes políticos"):
        st.image("https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTA60jNQ8axIwmKGqeJJZiwGHq9lT6z7o4hVA&s",use_container_width=False)
    with tab3:
      with st.expander("Usados en:"):
        objetivos = [
        "Sanidad.",
        "Ciberseguridad.",
        "Inteligencia artificial.",
        "Robótica."]
        for obj in objetivos:
          st.markdown(f"- {obj}")
      with st.expander("Casos de éxito"):
        st.image("https://i.ytimg.com/vi/C8SQE5EdflY/hq720.jpg?sqp=-oaymwEhCK4FEIIDSFryq4qpAxMIARUAAAAAGAElAADIQj0AgKJD&rs=AOn4CLACrn2scPkBOljsc0GGpMaKz0LShw",use_container_width=False)
        st.image("https://gomoot.com/wp-content/uploads/2025/08/deepseek3-1.webp",use_container_width=False)
        st.image("https://blog.cordatus.ai/wp-content/uploads/2025/07/phi4.webp",use_container_width=False)
    with tab4:
      with st.expander("Teorema de Sklar"):
        st.markdown("cualquier función de distribución conjunta multivariada puede ser escrita en términos de distribuciones marginales univariadas y una cópula, que describe la estructura de dependencia entre las variables.")
        formula_sklar = r'''H(x_1, x_2, \dots, x_n) = C(F_1(x_1), F_2(x_2), \dots, F_n(x_n))'''
        st.latex(formula_sklar)
      with st.expander("Ley fundamental de la recuperación de la información"):
        st.markdown("También conocidos como ataques de reconstrucción(reconstruction attacks, en inglés), afirma que demasiadas respuestas correctas a demasiadas preguntas destruyen la privacidad.")
      with st.expander("Pérdida de información"):
        st.markdown("Postulado por Rubin en 1993 y por Shannon en 1948, las categorías individuales y las relaciones menos probables tienden a desaparecer.")

elif seccion_seleccionada == "Estadística exploratoria y descriptiva":
    st.header("Estadística descriptiva y exploratoria")
    tab1,tab2,tab3= st.tabs(["Estructura de los datos","Datos faltantes por variable","Muestra seleccionada"])
    with tab2:
      data = {
        'Tipo': ['objeto', 'objeto', 'objeto', 'objeto', 'objeto', 'objeto',
             'int64', 'int64', 'int64', 'int64',
             'float64', 'float64', 'float64', 'float64',
             'float64', 'float64'],
      2016: [0, 0, 0, 0, 0, 0, 0, 0, 331, 331, 392, 392, 291, 291, 364, 364],
      2017: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 21, 21, 14, 14, 19, 19],
      2018: [0, 0, 0, 0, 0, 0, 81, 81, 0, 0, 103, 103, 132, 132, 82, 82],
      2019: [0, 0, 0, 0, 0, 0, 81, 81, 0, 0, 127, 127, 92, 92, 101, 101],
      2020: [0, 0, 0, 0, 0, 0, 216, 216, 0, 0, 187, 187, 252, 252, 183, 183]}
      index_labels = [
        'Apellidos Y Nombres', 'Grupo Referencia', 'Nombre Del Programa Académico',
        'Departamento institución', 'Municipio institución', 'Nombre De La Institución',
        'Percentil Comunicación Escrita', 'Puntaje Comunicación Escrita',
        'Percentil Razonamiento Cuantitativo', 'Puntaje Razonamiento Cuantitativo',
        'Percentil Lectura Crítica', 'Puntaje Lectura Crítica',
        'Percentil Competencias Ciudadanas', 'Puntaje Competencias Ciudadanas',
        'Percentil Inglés', 'Puntaje Inglés']
      df = pd.DataFrame(data, index=index_labels)
      df.index.name = 'Variable'
      df.columns.name = 'Año'
      st.markdown("Tipos de variables y valores faltantes por año: 2016-2020")
      st.caption("Tabla 1: Conteo de valores faltantes (NaN) por variable y por año en el conjunto de datos.")
      st.dataframe(df)
    with tab1:
      data = {
      'Año': [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024],
      'Observaciones': [11033, 10763, 10166, 10691, 12712, 14222, 11212, 10986, 13007],
      'Covariables': [16, 16, 16, 16, 16, 17, 17, 17, 17],
      'Datos': [176528, 172208, 162656, 171056, 203392, 241774, 190604, 186762, 221119],
      'Datos faltantes (%)': ['1.56', '0.06', '0.49', '0.47', '0.82', '0.00', '0.00', '0.00', '0.00'],
      '¿Imputar?': ['Sí', 'Sí', 'Sí', 'Sí', 'Sí', 'No', 'No', 'No', 'No']}

      df_resumen = pd.DataFrame(data)

      df_resumen['Datos faltantes (%)'] = df_resumen['Datos faltantes (%)'].str.replace(',', '.')


      df_resumen = df_resumen.set_index('Año')
      df_resumen.columns.name = None # Limpia el nombre de la columna superior
      st.markdown("Resumen de Datos Faltantes por Año (2016-2024)")
      st.caption("Tabla 2: Conteo de observaciones, covariables y porcentaje de datos faltantes en el conjunto total.")
      st.dataframe(df_resumen)
    with tab3:
      data = {
      'Año': [2016, 2017, 2018, 2019, 2020],
      'Evaluados': [245162, 248855, 240403, 269431, 253114],
      'Mejores': [11033, 10763, 10166, 10691, 12712],
      '% mejores': [4.5003, 4.3250, 4.2287, 3.9679, 5.0222]}
      # Crear el DataFrame
      df_estudiantes = pd.DataFrame(data)

      # Formatear el porcentaje para la visualización (opcional, pero mejora la lectura)
      df_estudiantes['% mejores'] = df_estudiantes['% mejores'].map('{:.4f}%'.format)

      # Establecer el 'Año' como índice (opcional)
      df_estudiantes = df_estudiantes.set_index('Año')
      df_estudiantes.columns.name = None

      # --- 2. Renderizar en Streamlit ---

      # Título de la tabla
      st.markdown("Estudiantes Evaluados (2016-2020)")
      st.caption("Tabla 3: estudiantes seleccionados como los 'Mejores' por el ICFES.")

      # Crear el DataFrame
      df_estudiantes = pd.DataFrame(data)

      # Formatear el porcentaje para la visualización (opcional, pero mejora la lectura)
      df_estudiantes['% mejores'] = df_estudiantes['% mejores'].map('{:.4f}%'.format)

      # Establecer el 'Año' como índice (opcional)
      df_estudiantes = df_estudiantes.set_index('Año')
      df_estudiantes.columns.name = None
      st.dataframe(df_estudiantes)

elif seccion_seleccionada == "Simulación de variables categóricas y entropía":
  tab1,tab2,tab3= st.tabs(["Simulación de variables categóricas","Medidas de calidad","Concepto de entropía"])
  with tab1:
    st.header("Simulación de variables categóricas")
    st.subheader("¿Cómo se simuló?")
    st.markdown("Siguiendo el teorema de Sklar, se capturó la estructura de dependencia entre variables. Para ello se realizó la simulación por medio de un árbol de probabilidad condicional supervisado.")
    url = f"https://docs.google.com/spreadsheets/d/18TbHDHJcjv9HZug3jEiW8RLio_RUdaCd/export?format=csv&gid=1388718316"
    st.dataframe(pd.read_csv(url),hide_index=True)
  with tab2:
    with st.expander("Medidas de similitud"):
     data = {
        'Variable': [
        'Grupo de Referencia',
        'Nombre del Programa Académico',
        'Departamento de la Institución',
        'Municipio de la Institución',
        'Nombre de la Institución'],
      'TCP (%)': ['12.6%', '2.88%', '17.7%', '21.31%', '3.03%'],
      'Chi2': [0.973, 1.000, 0.9995, 1.000, 1.000],
      'JS': [0.0071, 0.0412, 0.0099, 0.0128, 0.0267],
      'H': [0.0071, 0.0431, 0.0102, 0.0132, 0.0274]}
     df_calidad = pd.DataFrame(data)
     formatters = {
        'Chi2': '{:.4f}'.format,
        'JS': '{:.4f}'.format,
        'H': '{:.4f}'.format}
     st.caption("Tabla de calidad de las variables categóricas, donde se evalúan métricas de fidelidad.")
     st.dataframe(df_calidad.style.format(formatters), use_container_width=True, hide_index=True)
    with st.expander("Privacidad"):
      data = {
        'count': [55365.0, 55365.0],
        'mean': [0.478077, 0.521623],
        'std': [0.103485, 0.095807],
        'min': [0.000000, 0.115183],
        '25%': [0.448742, 0.473766],
        '50%': [0.492007, 0.506168],
        '75%': [0.526161, 0.550003],
        'max': [0.951634, 1.000000]}

      index_labels = [0, 1]  # Los valores de la columna 'Real'
      df_descripcion = pd.DataFrame(data, index=index_labels)
      df_descripcion.index.name = 'Real'
      st.dataframe(df_descripcion.style.format(precision=6), use_container_width=True)
  with tab3:
      st.header("Concepto de entropía")
      st.markdown("Medida de incertidubre, la cual puede ser mayor o menor en función de los estados posibles que esa variable pueda tomar.")
      with st.expander("Manifestación"):
       data = {
        'Variable': ['Depto. institución', 'Municipio institución', 'Institución', 'Programa académico', 'Grupo de referencia'],
        'Cat. Sintéticas': [27, 70, 252, 666, 23],
        'Cat. Reales': [27, 80, 266, 710, 23],
        'Cat. Perdidas': [0, 10, 14, 44, 0],
        'Moda (Tabla 1)': ['Bogotá', 'Bogotá, D. C.', 'Los Andes', 'Derecho', 'Ingeniería'],
        'Moda (Tabla 2)': ['Bogotá', 'Bogotá, D. C.', 'Los Andes', 'Derecho', 'Ingeniería']}
       df_comparativo = pd.DataFrame(data)
       st.dataframe(df_comparativo, use_container_width=True, hide_index=True)


elif seccion_seleccionada == "Simulación de variables numéricas y el trilema de los datos sintéticos":
  tab1,tab2,tab3,tab4= st.tabs(["Simulación de variables categóricas","Medidas de calidad","Trilema de los datos sintéticos","Utilidad"])
  with tab1:
    st.header("Simulación de variables numérica")
    st.subheader("¿Cómo se simuló?")
    st.markdown("Siguiendo el teorema de Sklar, se intentó capturar la estructura de dependencia entre variables. Se recurrió a cópulas gaussianas, cuyo proceso de construcción se automatizó con la librería SDV del MIT.")
    url = f"https://docs.google.com/spreadsheets/d/1bNg1I1oo0FKlLKUbjoS3XpY7Y-PJwIiJ/export?format=csv&gid=1261021413"
    st.dataframe(pd.read_csv(url),hide_index=True)
  with tab2:
    data = {
        'Variable': [
        'Grupo de Referencia',
        'Nombre del Programa Académico',
        'Departamento de la Institución',
        'Municipio de la Institución',
        'Nombre de la Institución'],
      'KS complemento': [0.9127, 0.9595, 0.9102, 0.9724, 0.9579],
      'JS': [0.0916, 0.0463, 0.1288, 0.0133, 0.1065],
      'H': [0.1002, 0.0466, 0.1316, 0.0135, 0.1181]}
    df_calidad = pd.DataFrame(data)
    formatters = {
        'Chi2': '{:.4f}'.format,
        'JS': '{:.4f}'.format,
        'H': '{:.4f}'.format}
    st.caption("Tabla de calidad de las variables categóricas, donde se evalúan métricas de fidelidad.")
    st.dataframe(df_calidad.style.format(formatters), use_container_width=True, hide_index=True)
  with tab3:
    with st.expander("Trilema de los datos sintéticos"):
      st.markdown("Flexibilidad, utilidad y privacidad son los pilares de la generación de datos sintéticos, pero producto de la ley fundamental de recuperación de información se deben sacrificar dos de las tres para que la propuesta funcione." )
  with tab4:

    resultados = pd.DataFrame({
      "Modelo": ["Referencia (Real)", "Prueba (Sintético)"],
      "MAE": [0.6754, 1.5698],
      "RMSE": [1.0635, 3.5409],
      "R2": [0.9927, 0.9195]})
    st.table(resultados)

    st.markdown('''
    - El modelo con datos reales alcanza un **R2 muy alto (0.993)**, indicando excelente ajuste.
    - El modelo con datos sintéticos tiene un **R2 de 0.92**, mostrando que conserva bastante la información pero con mayor error (MAE y RMSE más altos).
    ''')


elif seccion_seleccionada == "Referencias":
    st.header("Referencias")

    st.markdown('''
1. [Evaluating Synthetic Data: The Million-Dollar Question](https://towardsdatascience.com/evaluating-synthetic-data-the-million-dollar-question-a54701d1b621/)
2. [ScienceDirect: Evaluating synthetic data quality](https://www.sciencedirect.com/science/article/pii/S1386505624000765)
3. [Bluegen: How do I know that the synthetic data is of the right quality for my use case?](https://bluegen.ai/how-do-i-know-that-the-synthetic-data-is-of-the-right-quality-for-my-use-case/)
4. [UN News: Synthetic data and statistics](https://news.un.org/es/story/2018/11/1446671)
5. [UN: Universal Declaration of Human Rights](https://www.un.org/es/about-us/universal-declaration-of-human-rights#:~:text=Art%C3%ADculo%2012,contra%20tales%20injerencias%20o%20ataques.)
6. [ACL Anthology 2025](https://aclanthology.org/2025.cl-1.6.pdf)
7. [PMC Article on Synthetic Data](https://pmc.ncbi.nlm.nih.gov/articles/PMC9951365/)
8. [ScienceDirect Article](https://www.sciencedirect.com/science/article/pii/S2001037024002393)
9. [SCB: Statistical Analysis of Masked Data](https://www.scb.se/contentassets/ca21efb41fee47d293bbee5bf7be7fb3/statistical-analysis-of-masked-data.pdf)
10. [US Census Technical Paper](https://www2.census.gov/ces/tp/tp-2003-10.pdf)
11. [Royal Society: Synthetic Data Survey](https://royalsociety.org/-/media/policy/projects/privacy-enhancing-technologies/Synthetic_Data_Survey-24.pdf)
12. [Policy Review: Politics of Synthetic Data](https://policyreview.info/articles/news/politics-of-synthetic-data-performance-metrics/1761)
13. [UNESCO AI Ethics Recommendation](https://www.unesco.org/en/artificial-intelligence/recommendation-ethics)
14. [IBM AI Ethics](https://www.ibm.com/think/topics/ai-ethics#:~:text=Examples%20of%20AI%20ethics%20issues,%2C%20trust%2C%20and%20technology%20misuse.)
15. [NVIDIA Glossary: Synthetic Data Generation](https://www.nvidia.com/en-us/glossary/synthetic-data-generation/)
16. [DataCamp Tutorial: Synthetic Data Generation](https://www.datacamp.com/tutorial/synthetic-data-generation)
17. [SDV: Synthetic Data Vault](https://sdv.dev/)
''', unsafe_allow_html=True)
"""
# Guardar el código Streamlit en un archivo
with open("/content/streamlit_plotly_timeseries.py", "w") as f:
    f.write(streamlit_code)

# Establecer el token de autenticación de ngrok
from pyngrok import ngrok
ngrok.set_auth_token("2tzdV3oofirn68JeCgolauxDQUt_75mTMJ3ko2AKPPnGaxi9A")

# Crear el túnel de ngrok a Streamlit en el puerto 8501
public_url = ngrok.connect(8501)
print(f"Streamlit app is live at: {public_url}")

# Ejecutar la aplicación Streamlit en segundo plano
!streamlit run /content/streamlit_plotly_timeseries.py &

<>:301: SyntaxWarning: invalid escape sequence '\d'
<>:301: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_5102/909307958.py:301: SyntaxWarning: invalid escape sequence '\d'
  formula_sklar = r'''H(x_1, x_2, \dots, x_n) = C(F_1(x_1), F_2(x_2), \dots, F_n(x_n))'''


Streamlit app is live at: NgrokTunnel: "https://07cd-34-125-93-7.ngrok-free.app" -> "http://localhost:8501"



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.93.7:8501



#Referencias

##Medidas de calidad
* https://towardsdatascience.com/evaluating-synthetic-data-the-million-dollar-question-a54701d1b621/
* https://www.sciencedirect.com/science/article/pii/S1386505624000765
* https://bluegen.ai/how-do-i-know-that-the-synthetic-data-is-of-the-right-quality-for-my-use-case/
* https://news.un.org/es/story/2018/11/1446671
* https://www.un.org/es/about-us/universal-declaration-of-human-rights#:~:text=Art%C3%ADculo%2012,contra%20tales%20injerencias%20o%20ataques.
* https://aclanthology.org/2025.cl-1.6.pdf
* https://pmc.ncbi.nlm.nih.gov/articles/PMC9951365/
* https://www.sciencedirect.com/science/article/pii/S2001037024002393
*   https://www.scb.se/contentassets/ca21efb41fee47d293bbee5bf7be7fb3/statistical-analysis-of-masked-data.pdf
*   https://www2.census.gov/ces/tp/tp-2003-10.pdf
*   https://royalsociety.org/-/media/policy/projects/privacy-enhancing-technologies/Synthetic_Data_Survey-24.pdf
*   https://policyreview.info/articles/news/politics-of-synthetic-data-performance-metrics/1761
* https://www.unesco.org/en/artificial-intelligence/recommendation-ethics
* https://www.ibm.com/think/topics/ai-ethics#:~:text=Examples%20of%20AI%20ethics%20issues,%2C%20trust%2C%20and%20technology%20misuse.
* https://www.nvidia.com/en-us/glossary/synthetic-data-generation/
* https://www.datacamp.com/tutorial/synthetic-data-generation
* https://sdv.dev/